In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

TODO:

- Try class for `Card` later? `namedtuple` works for now.
- Try `IntEnum` for Poker hand rankings. Currently, we use integer values from 14 to 1 to indicate the poker hand rankings. Using `IntEnum` may make it more readable. Read more about enum: https://docs.python.org/3/library/enum.html
- Try passing `Round` to `Player` when taking action to update `stage_bet`.
- Player should not be able to check if `stage_bet != player.bet`.

In [ ]:
from fastcore.utils import *

# core

> Fill in a module description here

In this dialog, I want to go through the process of writing a poker program myself but following the approach I learned in the cs212 class. I went through the lesson in `pytude/cs212/lesson_1` dialog. Read the dialog to guide me when I ask for help without giving me answers directly. It's been some time since I went over this lesson, so I mostly likely forgot about all the details of implementations. My goal is not to copy everything from the lesson exactly, but follow the steps: understanding the problem, specifying, concept inventory, etc. that I can apply to solving any problem. To challenge myself, I will write texas hold'em poker.

Here's the process Norvig taught:

**1. Understand** — Start with a vague understanding and refine it. Make an inventory of all the concepts you'll need to deal with (data types, operations, etc.).

**2. Specify** — Define *what* needs to happen: inputs, outputs, what each function should accomplish. Think of it as a contract — not the implementation details, just the "what."

**3. Design** — Figure out *how* to make it happen — the actual algorithms, code structure, and implementation.

Along the way, key principles:
- **Write tests early** (even before implementation), including **extreme values**
- **Reuse existing tools** when possible
- **Refactor** for elegance (DRY, clarity, simplicity, generality)
- Be aware of the **four dimensions**: correctness, efficiency, features, elegance — and make conscious tradeoffs

## Understanding the problem.

Rules in the poker:

Poker is a card game with 52 cards, and each card has a suit and a rank. A suit consists of spades, clubs, hearts, and diamonds. A rank consists of numbers from ace, 2 to 10, jack, queen, and king. A hand has two cards, and community cards have five cards. They are combined to make different combinations used for comparisons among hands. At the start of the round, each player gets two cards face down and three cards in the community. After each betting round, one additional card is dealt to the community until there are five cards. Each hand from the players is compared, and winner takes all the bets unless there is a tie, and they split. To compare the hands, poker ranking is used. At each betting round, players can call, check, raise, or fold. If everyone folds except for one player, that player wins. If a player raises, others have to match the bet (call) or raise, or fold. After everyone checks, next round starts. 

Poker hands:
- royal flush
- straight flush
- four of a kind
- full house
- flush
- straight
- three of a kind
- two pair
- one pair
- high card

Concept inventory

Core concepts:
  - `Rank`: 2 to 10, jack, queen, king, and ace
  - `suit`: diamonds, hearts, clubs, spades.
  - `Card`: with rank and suit.
  - `CommunityCards`: Three to five cards shared by all the players.
  - `Hand`: 2 hole cards.
  - `HandRank`: Combining `Hand` with `CommunityCards`, hand rankings are compared among the players. From 7 cards, best 5 cards are used.
  - `Stage`: A stage is where each player takes actions such as call, fold, raise, and check. It ends when everyone folds except for those who called.
  - `Round`: A round consists of four stages (pre-flop, after flop, turn, and river). 
  - `Stage_bet`: money to match each round for each player and becomes the prize pool.
  - `Bet`: Money each player put in this stage.
  - `Balance`: The amount of moeny each player possesses so they can use for betting.
  - `Player state`: Useful player state such as balance, hand, whether playing the round or not, and bet.
  - `Deck`: 52 cards that get shuffled.
  - `Pot`: prize money that gets distributed to winners at the end of each round.

Add-ons:
  - `Blinds`: small and big blind. Small is immediately to the left of the dealer, and big blind is after. Big blind is twice the small blind amount.
  - `Dealer`: rotates clockwise each hand.
  - `display`: text based or graphical display.
  - `GameObserver/AI advisor`: sees the public game state and hole cards. Answers questions on request.

Operations

- Shuffle the deck
- Deal cards
- Perform player actions: call, fold, check, raise
- Evaluate a hand rank
- Compare hand ranks
- Distribute the pot to winners
- Start/end rounds
- Rotate dealer
- Draw community cards

**Specification**

Specifying is the step where you define **what** your program should do, without worrying about *how* it does it. It's like writing a contract:

- What are the **inputs**?
- What are the **outputs**?
- What **behavior** should the program have?

--- 

Card & Deck Operations
- `shuffle(deck) -> None`: Shuffle the deck.
- `deal_hole(deck, players) -> None`: Deal 2 cards to each player.
- `deal_community(deck, community_cards, n) -> None`: Deal n cards from deck to community cards.

Hand Evaluation
  - `evaluate_hand(hand, community_cards) -> hand_rank`: From 7 cards, find the best 5-card hand and return its rank.
  - `compare_ranks(hand_ranks) -> tuple`: Given a set of hand ranks, return the best one.

Player Actions
  - `perform_action(player, round) -> None`: Prompt player for an action, then call the appropriate function.
  - `call(player, round) -> None`: Player matches the current bet. Round state updated.
  - `fold(player, round) -> None`: Player folds. Round state updated.
  - `check(player, round) -> None`: Player checks. Round state updated.
  - `raise(player, round, amount) -> None`: Player raises by amount. Round state updated.

Round Management
  - `betting_round(round) -> None`: Manage one round of player actions (pre-flop, flop, turn, or river). Ends when all active players have called or checked.
  - `start_round(round) -> None`: Orchestrate the whole hand: deal hole cards → blinds → betting round → flop → betting round → turn → betting round → river → betting round → showdown → distribute.
  - `distribute(pot, winners) -> None`: Distribute the pot to the winner(s).

State
- round: tracks players with their bets, pot, community cards, and current betting state.
- player: has balance, current bet, hand, and whether folded or active.

**Design**

Design is where you figure out **how** to make it happen. The spec said *what* your program does; design says *how* it does it.

This is where you make decisions like:

- **Data representation** — How do you represent a card? A tuple `(rank, suit)`? A string `"AS"`? A custom class?
- **Code structure** — How do you organize the functions? What goes in a class vs standalone functions?
- **Algorithms** — How do you evaluate the best 5-card hand from 7? How do you determine when a betting round is over?

It's still not about writing the final code — it's about making the key implementation choices before you start coding.


`Card`:
- In the beginning, I tried using a string "♠2". This was a very simple and easy way to represent a card. To grab a rank or a suit from a card, `c1`, I can do `c1[0]` or `c1[1:]`. To make it more readable, I defined a function for these operations. Then I thought it was more convenient to use a dot notation. Rather than using a class, I tried using `namedtuple` from `collections`. This was a very light weight.

class `Player` with attributes 
- balance: int
- bet: int
- playing: bool
- hand: List[Card]
methods: 
- take_action(round)
- call(round)
- check(round)
- fold(round)
- raise(round, amount). 

All the methods need to pass `round` and update it. Therefore, I might get a `round` attribute. But this leads to circular dependency, which tightly couples these together. It becomes harder to test.

`Round` class attributes are 
- players: List[Player]
- dealer: int
- deck: Deck
- stage: Enum(pre-flop, flop, turn, river)
- pot: int
- community: CommunityCard

methods:
- betting_round()
- start_round()
- distribute()

**Architecture**

`Game` is the top-level mediator. It owns `players` and the current `round`. Neither `Player` nor `Round` reference each other directly — `Game` coordinates between them.

**State ownership**

`Round` is the single owner of all mutable state: player balances, bets, pot, community cards, dealer position, and stage. All state updates flow through `Round`.

**Functional data operations**

`Deck` and `Card` operations are pure — they return new values rather than mutating:

- `shuffle() -> Deck` — returns a new shuffled deck
- `deal(n) -> (cards, Deck)` — returns dealt cards and a new deck with remaining cards

**Pure logic functions**

Hand evaluation and bet calculations:

- `evaluate_hand(hand, community) -> HandRank` — given 7 cards, returns the best 5-card rank
- `compare_ranks(ranks) -> tuple` — returns the winner rank

**Round methods**

`Round` applies results from pure functions to update state:

- `betting_round()` — manages one round of actions
- `start_round()` — orchestrates the full hand
- `distribute(winners)` — distributes pot to winners

**Player actions**

Players receive only the data they need (current bet, pot, etc.) rather than a full `Round` reference. Actions return *what* the new state should be, and `Round` applies it.

How to compare hands? Let's use a tuple with first entry representing the `hand_rank` from royal flush to hgih card with 10 to 1. Then later entries for tie breakers.

Poker hands:
- royal flush (10)
- straight flush (9, highest_value)
- four of a kind (8, highest_value, kicker)
- full house (7, triple, pair)
- flush (6, five kickers)
- straight (5, highest_value)
- three of a kind (4, triple_value, two kickers)
- two pair (3, high_pair, low_pair, kicker)
- one pair (2, pair_value, three kickers)
- high card (1, highest, second highest, third, fourth, fifth)

## Card

There are many ways to design `Card`. One of the simplest way is to use string.

In [ ]:
Card = str

I use emoji for suits and string of alphanumerics for ranks.

In [ ]:
suits = '♠♥♦♣'
suits

'♠♥♦♣'

In [ ]:
ranks = '2,3,4,5,6,7,8,9,10,J,Q,K,A'.split(',')
ranks

['2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K', 'A']

In [ ]:
Card(suits[0] + ranks[-1])

'♠A'

Rather than using a string for cards and using separate functions to access suits and ranks, I thought it would be better to use the `.` (dot) notation. Rather than using a `class`, I wanted to try using `namedtuple`, which allows me to use attribute access and immutability. The ranks and suits from each card should not change. It is more lightweight than `class`.

In [ ]:
from collections import namedtuple
Card = namedtuple('Card', ['suit', 'rank'])
Card

__main__.Card

In [ ]:
c1 = Card('♠', '10')
c1.suit, c1.rank

('♠', '10')

In [ ]:
c1

Card(suit='♠', rank='10')

I want the card representation to be more concise, so I patch it.

In [ ]:
@patch
def __repr__(self: Card): return self.suit + self.rank

In [ ]:
c2 = Card('♠', 'K')
c2

♠K

Currently, we cannot compare the ranks directly as we are using string comparisons lexicographically. 

In [ ]:
c1.rank > c2.rank, '10' > 'K'

(False, False)

It is convenient to sort hands by rank values when evaluating hands. 

Using `rank_values` mapping, we can convert the string ranks into integers. From 2 to 10, their values are 2 to 10 respectively. Then, from 'J' to 'A', 11 to 14.

In [ ]:
rank_values = {r:i for i,r in enumerate(ranks, start=2)}
rank_values

{'2': 2,
 '3': 3,
 '4': 4,
 '5': 5,
 '6': 6,
 '7': 7,
 '8': 8,
 '9': 9,
 '10': 10,
 'J': 11,
 'Q': 12,
 'K': 13,
 'A': 14}

Although I had `sort_cards` function that sorts cards by their `rank_values` from the highest, it was more convenient to patch `__lt__` and `__eq__` and just use default `sorted` function. When comparing cards, we only care about their ranks, not ranks. 

In [ ]:
@patch
def __lt__(self: Card, other: Card): return rank_values[self.rank] < rank_values[other.rank]

In [ ]:
@patch
def __eq__(self: Card, other: Card): return rank_values[self.rank] == rank_values[other.rank]

In [ ]:
deck = [Card(s,r) for s in suits for r in ranks]
sorted(deck,reverse=True)[:5]

[♠A, ♥A, ♦A, ♣A, ♠K]

## Hand

- `evaluate_hand(cards) -> HandRank` — given 7 cards, returns the best 5-card `HandRank`
- `compare_ranks(ranks) -> tuple` — returns the winner rank

In [ ]:
from collections import Counter

A hand has two cards. Community cards have five cards. 

Poker hands:
- royal flush (10)
- straight flush (9, highest_value)
- four of a kind (8, highest_value, kicker)
- full house (7, triple, pair)
- flush (6, five kickers)
- straight (5, highest_value)
- three of a kind (4, triple_value, two kickers)
- two pair (3, high_pair, low_pair, kicker)
- one pair (2, pair_value, three kickers)
- high card (1, highest, second highest, third, fourth, fifth)

In [ ]:
c1, c2

(♠10, ♠K)

In [ ]:
c1.rank

'10'

### Sample hands

These are some example hands.

In [ ]:
cards_highcard = [Card(suits[0], ranks[0]), Card(suits[1], ranks[2]), Card(suits[2], ranks[4]),
    Card(suits[3], ranks[6]), Card(suits[0], ranks[8]), Card(suits[1], ranks[10]), Card(suits[2], ranks[12])]
cards_highcard

[♠2, ♥4, ♦6, ♣8, ♠10, ♥Q, ♦A]

In [ ]:
cards_2kind = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[0], ranks[3]), Card(suits[1], ranks[5]), 
    Card(suits[2], ranks[7]), Card(suits[3], ranks[9]), Card(suits[0], ranks[11])]
cards_2kind

[♠2, ♥2, ♠5, ♥7, ♦9, ♣J, ♠K]

In [ ]:
cards_3kind = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[2], ranks[0]), Card(suits[0], ranks[3]), 
    Card(suits[1], ranks[5]), Card(suits[2], ranks[7]), Card(suits[3], ranks[9])]
cards_3kind

[♠2, ♥2, ♦2, ♠5, ♥7, ♦9, ♣J]

In [ ]:
cards_4kind = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[2], ranks[0]), Card(suits[3], ranks[0]),
    Card(suits[0], ranks[4]), Card(suits[1], ranks[7]), Card(suits[2], ranks[10])]
cards_4kind

[♠2, ♥2, ♦2, ♣2, ♠6, ♥9, ♦Q]

In [ ]:
cards_2pair = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[2], ranks[1]), Card(suits[3], ranks[1]),
    Card(suits[0], ranks[4]), Card(suits[1], ranks[6]), Card(suits[2], ranks[8])]
cards_2pair

[♠2, ♥2, ♦3, ♣3, ♠6, ♥8, ♦10]

In [ ]:
cards_fullhouse = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[0], ranks[1]), Card(suits[2], ranks[1]), 
    Card(suits[3], ranks[1]), Card(suits[1], ranks[6]), Card(suits[2], ranks[8])]
cards_fullhouse

[♠2, ♥2, ♠3, ♦3, ♣3, ♥8, ♦10]

In [ ]:
cards_straight = [Card(suits[0], ranks[1]), Card(suits[1], ranks[2]), Card(suits[2], ranks[3]),
    Card(suits[3], ranks[4]), Card(suits[0], ranks[5]), Card(suits[1], ranks[8]), Card(suits[2], ranks[10])]
cards_straight

[♠3, ♥4, ♦5, ♣6, ♠7, ♥10, ♦Q]

In [ ]:
cards_flush = [Card(suits[0], ranks[0]), Card(suits[0], ranks[2]), Card(suits[0], ranks[4]),
    Card(suits[0], ranks[6]), Card(suits[0], ranks[8]), Card(suits[1], ranks[3]), Card(suits[2], ranks[7])]
cards_flush

[♠2, ♠4, ♠6, ♠8, ♠10, ♥5, ♦9]

In [ ]:
cards_straightflush = [Card(suits[0], ranks[1]), Card(suits[0], ranks[2]), Card(suits[0], ranks[3]),
    Card(suits[0], ranks[4]), Card(suits[0], ranks[5]), Card(suits[1], ranks[8]), Card(suits[2], ranks[10])]
cards_straightflush

[♠3, ♠4, ♠5, ♠6, ♠7, ♥10, ♦Q]

In [ ]:
cards_royalflush = [Card(suits[0], ranks[12]), Card(suits[0], ranks[11]), Card(suits[0], ranks[10]), 
    Card(suits[0], ranks[9]), Card(suits[0], ranks[8]), Card(suits[1], ranks[2]), Card(suits[2], ranks[4])]
cards_royalflush

[♠A, ♠K, ♠Q, ♠J, ♠10, ♥4, ♦6]

### Ranks and counts

By getting the counts from the cards, we can figure out `four of a kind`, `full house`, `three of a kind`, `two pair`, and `one pair`. One thing we have to be careful of is because we have seven cards to evaluate, `full house` might have two `three of kind`s, rather than a `three of kind` and a `pair`.

Here is `get_ranks_counts`, which takes a list of cards and returns a tuple of ranks and counts. The output is sorted by the rank counts and `rank_values` for tie breakers. Rank counts are useful for finding what kind of poker ranking hand it is, and ranks are used for the main hand components and kickers.

In [ ]:
def get_ranks_counts(cards: list[Card]) -> tuple[tuple[int, ...], tuple[int, ...]]:
    """Return ((ranks...), (counts...)) sorted by count then by rank value from cards"""
    items = Counter([rank_values[c.rank] for c in cards]).items()
    return tuple(zip(*sorted(items, key=lambda o: (o[1], o[0]), reverse=True)))

In [ ]:
get_ranks_counts(cards_fullhouse)

((3, 2, 10, 8), (3, 2, 1, 1))

In [ ]:
def test_get_ranks_counts():
    assert get_ranks_counts(cards_2kind) == ((2, 13, 11, 9, 7, 5), (2, 1, 1, 1, 1, 1))
    assert get_ranks_counts(cards_3kind) == ((2, 11, 9, 7, 5), (3, 1, 1, 1, 1))
    assert get_ranks_counts(cards_4kind) == ((2, 12, 9, 6), (4, 1, 1, 1))
    assert get_ranks_counts(cards_2pair) == ((3, 2, 10, 8, 6), (2, 2, 1, 1, 1))
    assert get_ranks_counts(cards_fullhouse) == ((3, 2, 10, 8), (3, 2, 1, 1))
    assert get_ranks_counts(cards_highcard) == ((14, 12, 10, 8, 6, 4, 2), (1, 1, 1, 1, 1, 1, 1))
    assert get_ranks_counts(cards_straight) == ((12, 10, 7, 6, 5, 4, 3), (1, 1, 1, 1, 1, 1, 1))

    # Edge case: two three-of-a-kinds (e.g. 222 333 K) — full house should pick higher triple
    cards_2trips = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[2], ranks[0]),
                    Card(suits[0], ranks[1]), Card(suits[1], ranks[1]), Card(suits[2], ranks[1]),
                    Card(suits[3], ranks[11])]
    assert get_ranks_counts(cards_2trips) == ((3, 2, 13), (3, 3, 1))

test_get_ranks_counts()

### flush

As long as the suit count is 5 or more, it is flush because we have seven cards total.

In [ ]:
def is_flush(cards: list[Card]) -> list:
    "If cards have flush, return the `rank_values` from the cards in sorted order. If not, return []"
    suit, count = Counter(c.suit for c in cards).most_common()[0]
    if count >= 5:
        return sorted([rank_values[c.rank] for c in cards if c.suit == suit][:5], reverse=True)
    return []

In [ ]:
def test_is_flush():
    # Flush: 5 spades, two side cards
    assert is_flush(cards_flush) == [10, 8, 6, 4, 2]

    # Straight flush
    assert is_flush(cards_straightflush) == [7, 6, 5, 4, 3]

    # Royal flush
    assert is_flush(cards_royalflush) == [14, 13, 12, 11, 10]

    # No flush: suits too spread out
    assert is_flush(cards_fullhouse) == []
    assert is_flush(cards_highcard) == []
    assert is_flush(cards_4kind) == []

    # Edge: all 7 cards same suit — returns top 5
    all_spades = [Card('♠', r) for r in 'A,K,Q,J,10,9,8'.split(',')]
    assert is_flush(all_spades) == [14, 13, 12, 11, 10]

    # Edge: exactly 5 same suit
    five_hearts = [Card('♥', r) for r in '2,3,4,5,6'.split(',')] + [Card('♠', 'A'), Card('♦', 'K')]
    assert is_flush(five_hearts) == [6, 5, 4, 3, 2]

test_is_flush()

### straight

Finding straight is trickier because Ace can serve as 1 or 14. Also, there are seven cards, but only five are used. 

In [ ]:
def is_straight(cards: list[Card]) -> int:
    "Return the highest value of the straight if cards contain 5 consecutive ranks, else 0"
    vals = sorted([rank_values[c.rank] for c in cards], reverse=True)
    if vals[0] == 14: 
        vals.append(1)
    for i in range(len(vals) - 4):
        if vals[i] - vals[i+4] == 4 and len(set(vals[i:i+5])) == 5:
            return vals[i]
    return 0

In [ ]:
def test_is_straight():
    # Regular straight: 3-4-5-6-7
    assert is_straight(cards_straight) == 7
    
    # Ace-high straight (10-J-Q-K-A)
    ace_high = [Card('♠','10'), Card('♥','J'), Card('♦','Q'), Card('♣','K'), Card('♠','A'),
                Card('♥','3'), Card('♦','6')]
    assert is_straight(ace_high) == 14
    
    # Wheel: A-2-3-4-5 (ace low)
    wheel = [Card('♠','A'), Card('♥','2'), Card('♦','3'), Card('♣','4'), Card('♠','5'),
             Card('♥','9'), Card('♦','J')]
    assert is_straight(wheel) == 5
    
    # No straight — gapped
    assert is_straight(cards_2pair) == 0
    
    # No straight — bunch of duplicates that run together in rank
    dupes = [Card('♠','5'), Card('♥','5'), Card('♦','6'), Card('♣','6'), Card('♠','7'),
             Card('♥','7'), Card('♦','8')]
    assert is_straight(dupes) == 0

test_is_straight()

### evaluate_hand

Now that we can detect flush and straight, we can update `evaluate_hand`.

In [ ]:
def evaluate_hand(cards: list[Card]) -> tuple:
    """Return hand rank tuple for the best 5-card hand from `cards`.
    Hands ranked from 1 (high card) to 10 (royal flush). Each rank tuple
    starts with the hand category, followed by tiebreaker values (kickers
    or relevant ranks).
    """
    ranks, counts = get_ranks_counts(cards)
    straight_val = is_straight(cards)
    flush_vals = is_flush(cards)
    if straight_val == 14: # royal flush
        return (10,)
    elif straight_val != 0 and flush_vals != []: # straight flush
        return (9, straight_val)
    elif counts[0] == 4: # four of a kind
        return (8, ranks[0], ranks[1])
    elif counts[0] == 3 and counts[1] >= 2: # full house
        return (7, ranks[0], ranks[1])
    elif flush_vals != []: # flush
        return (6, *flush_vals)
    elif straight_val != 0: # straight
        return (5, straight_val)
    elif counts[0] == 3: # three of a kind
        return (4, ranks[0], ranks[1], ranks[2])
    elif counts[0] == 2 and counts[1] == 2: # two pair
        return (3, ranks[0], ranks[1], ranks[2])
    elif counts[0] == 2: # one pair
        return (2, ranks[0], ranks[1], ranks[2], ranks[3])
    else: # high card
        return (1, *ranks)

In [ ]:
def test_evaluate_hand():
    # Royal flush: A♠ K♠ Q♠ J♠ 10♠ → (10,)
    assert evaluate_hand(cards_royalflush) == (10,)

    # Straight flush: 3♠ 4♠ 5♠ 6♠ 7♠ → (9, 7)
    assert evaluate_hand(cards_straightflush) == (9, 7)

    # Four of a kind: 2222 + Q 9 6 → (8, 2, 12)
    assert evaluate_hand(cards_4kind) == (8, 2, 12)

    # Full house: 333 22 + 10 8 → (7, 3, 2)
    assert evaluate_hand(cards_fullhouse) == (7, 3, 2)

    # Flush: 5 spades → (6, 10, 8, 6, 4, 2)
    assert evaluate_hand(cards_flush) == (6, 10, 8, 6, 4, 2)

    # Straight: 3-4-5-6-7 → (5, 7)
    assert evaluate_hand(cards_straight) == (5, 7)

    # Three of a kind: 222 + J 9 7 5 → (4, 2, 11, 9)
    assert evaluate_hand(cards_3kind) == (4, 2, 11, 9)

    # Two pair: 33 22 + 10 8 6 → (3, 3, 2, 10)
    assert evaluate_hand(cards_2pair) == (3, 3, 2, 10)

    # One pair: 22 + K J 9 7 5 → (2, 2, 13, 11, 9)
    assert evaluate_hand(cards_2kind) == (2, 2, 13, 11, 9)

    # High card: A Q 10 8 6 4 2 → (1, 14, 12, 10, 8, 6, 4, 2)
    assert evaluate_hand(cards_highcard) == (1, 14, 12, 10, 8, 6, 4, 2)

test_evaluate_hand()

### compare_hands

`compare_hands(hands: List[List[Card]]) -> tuple` — returns the ranks for the winner(s)

Winners are those who have the winning ranks

In [ ]:
def compare_hands(hands: list[list[Card]]) -> tuple:
    """Return the winning hand rank among `hands`."""
    return sorted([evaluate_hand(h) for h in hands], reverse=True)[0]

In [ ]:
evaluate_hand(cards_flush)

(6, 10, 8, 6, 4, 2)

In [ ]:
def test_compare_hands():
    # Clear winner: flush beats one pair
    assert compare_hands([cards_2kind, cards_flush]) == (6, 10, 8, 6, 4, 2)
    
    # Tie: two identical hands → both should win
    assert compare_hands([cards_flush, cards_flush]) == (6, 10, 8, 6, 4, 2)
    
    # Three hands, two tie for best
    assert compare_hands([cards_2kind, cards_flush, cards_royalflush]) == (10,)
    
    # Reverse order shouldn't matter
    assert compare_hands([cards_flush, cards_2kind]) == (6, 10, 8, 6, 4, 2)

test_compare_hands()

## Deck

In my original plan, I thought of writing a `CardCollection` class as an abstract class and create `Hand`, `Deck`, and `CommunityCards` by inheriting from it as they all consist of cards. However, they don't really have methods that share. To keep things simple, I decided to start implementing `Deck` as a list of `Card`s and see how it feels.

A deck consists of 52 cards. From that, we need to shuffle and withdraw cards so the dealer can deal cards.

In [ ]:
import random

In [ ]:
deck = [Card(s,r) for s in suits for r in ranks]
deck[:5]

[♠2, ♠3, ♠4, ♠5, ♠6]

In [ ]:
def shuffle_deck(deck: list[Card]) -> list[Card]:
    "Shuffle `deck` in place and return it"
    random.shuffle(deck)
    return deck

In [ ]:
random.seed(42)

deck = [Card(s,r) for s in suits for r in ranks]
shuffle_deck(deck)[:5]

[♠J, ♥Q, ♥A, ♠5, ♥10]

In [ ]:
def withdraw_card(deck: list[Card], n: int=1) -> list[Card]:
    "Withdraw `n` cards from `deck` and return them. Modifies `deck` in place."
    return [deck.pop() for _ in range(n)]

In [ ]:
deck = [Card(s,r) for s in suits for r in ranks]
withdraw_card(deck)

[♣A]

In [ ]:
assert len(deck) == 51

In [ ]:
withdraw_card(deck, n=5)
assert len(deck) == 46

Can you create a small deck with five cards to teach and demonstrate me how probability and combinatorics work? Use python tool.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let's start with a small 5-card deck to make counting easy. Here's a simple one:

```python
small_deck = ['♠A', '♠K', '♠Q', '♠J', '♠10']
small_deck
```



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;small_deck = [&#x27;♠A&#x27;, &#x27;♠K&#x27;, &#x27;♠Q&#x27;, &#x27;♠J&#x27;, &#x27;♠…&quot;)→&quot;[&#x27;♠A&#x27;, &#x27;♠K&#x27;, &#x27;♠Q&#x27;, &#x27;♠J&#x27;, &#x27;♠10&#x27;]&quot;</code></summary>

```json
{
  "id": "call_00_CWAzLIuzWqfZIX57qO3h2859",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "small_deck = ['♠A', '♠K', '♠Q', '♠J', '♠10']\nsmall_deck"
    }
  },
  "result": "['♠A', '♠K', '♠Q', '♠J', '♠10']"
}
```

</details>





We have 5 cards: ♠A, ♠K, ♠Q, ♠J, ♠10.

Let's start with a simple question: **how many ways can you draw 2 cards** (order doesn't matter)? We can list them:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;from itertools import combinations\n\nfor …&quot;)→&quot;(&#x27;♠A&#x27;, &#x27;♠K&#x27;)\n(&#x27;♠A&#x27;, &#x27;♠Q&#x27;)\n(&#x27;♠A&#x27;, &#x27;♠J&#x27;)\n(…&quot;</code></summary>

```json
{
  "id": "call_00_cLDe5sduTyvCnwKFGApD2340",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "from itertools import combinations\n\nfor combo in combinations(small_deck, 2):\n    print(combo)\nprint(f\"\\nTotal: {len(list(combinations(small_deck, 2)))} combinations\")"
    }
  },
  "result": "('♠A', '♠K')\n('♠A', '♠Q')\n('♠A', '♠J')\n('♠A', '♠10')\n('♠K', '♠Q')\n('♠K', '♠J')\n('♠K', '♠10')\n('♠Q', '♠J')\n('♠Q', '♠10')\n('♠J', '♠10')\n\nTotal: 10 combinations"
}
```

</details>





10 combinations — that's \( \binom{5}{2} = \frac{5 \times 4}{2} = 10 \).

Now here's a question for you: if you draw 2 cards at random from this deck, what's the probability that **one of them is ♠A**? 

Take a look at the list above and see if you can figure it out before we calculate it with code.

<details class='token-usage-details' markdown='1'><summary>$0.0156</summary>

`total=124,440 | in=123,872 | out=568 | cached=90.7% | reasoning=205 | $0.0156 | deepseek-v4-pro`

</details>


Can you create a small deck with five cards to teach and demonstrate me how probability and combinatorics work? Use python tool.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good idea! Let me set up a tiny deck and explore some probability basics with it.



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;# A mini deck of 5 cards — small enough …&quot;)→&quot;[♠A, ♠K, ♠Q, ♥A, ♦2]&quot;</code></summary>

```json
{
  "id": "call_00_OetJdzEmdJeDQAiBjKOJ6885",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "# A mini deck of 5 cards — small enough to enumerate all possibilities by hand\nmini_deck = [Card('♠','A'), Card('♠','K'), Card('♠','Q'), Card('♥','A'), Card('♦','2')]\nmini_deck"
    }
  },
  "result": "[♠A, ♠K, ♠Q, ♥A, ♦2]"
}
```

</details>





Here's our mini-deck: **♠A, ♠K, ♠Q, ♥A, ♦2**

5 cards. Small enough to count everything by hand, but rich enough to show real poker concepts.

Let me start with a simple question:

**If I draw one card at random, what's the probability of...**

- Drawing an Ace?    (2 Aces out of 5 → 2/5 = 40%)
- Drawing a spade?   (3 spades out of 5 → 3/5 = 60%)
- Drawing the Ace of spades? (1 card out of 5 → 1/5 = 20%)

Those are straightforward. Now a more interesting one — **combinations**:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;from itertools import combinations\n\n# Ho…&quot;)→&quot;10&quot;</code></summary>

```json
{
  "id": "call_00_b12LN30PniCWETnJ9C1q4588",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "from itertools import combinations\n\n# How many ways to draw 2 cards from the mini-deck?\ntwo_card_hands = list(combinations(mini_deck, 2))\nlen(two_card_hands)"
    }
  },
  "result": "10"
}
```

</details>





10 possible 2-card hands. Let's list them:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;for i, hand in enumerate(two_card_hands,…&quot;)→&quot; 1. ♠A  ♠K\n 2. ♠A  ♠Q\n 3. ♠A  ♥A\n 4. ♠A …&quot;</code></summary>

```json
{
  "id": "call_00_oyZx2MfsTtISL1qkXDtF2246",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "for i, hand in enumerate(two_card_hands, 1):\n    print(f\"{i:2d}. {hand[0]}  {hand[1]}\")"
    }
  },
  "result": " 1. ♠A  ♠K\n 2. ♠A  ♠Q\n 3. ♠A  ♥A\n 4. ♠A  ♦2\n 5. ♠K  ♠Q\n 6. ♠K  ♥A\n 7. ♠K  ♦2\n 8. ♠Q  ♥A\n 9. ♠Q  ♦2\n10. ♥A  ♦2"
}
```

</details>





Here are all 10 possible 2-card hands (order doesn't matter). From these, we can compute probabilities like:

- **Pocket Aces** (♠A ♥A): only hand #3 → 1/10 = 10%
- **At least one Ace**: hands #1,2,3,4,6,8,10 → 7/10 = 70%
- **Suited** (both spades): hands #1,2,5 → 3/10 = 30%
- **Offsuit** (different suits): everything else → 7/10 = 70%

Now let's check something you'll care about in Texas Hold'em: **what about drawing 3 cards?** (Like community cards.)



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;three_card_combos = list(combinations(mi…&quot;)→&quot;Total 3-card combos: 10\nTotal 3-card com…&quot;</code></summary>

```json
{
  "id": "call_00_tHHquM7R8NmoNkCjUulr4039",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "three_card_combos = list(combinations(mini_deck, 3))\nprint(f\"Total 3-card combos: {len(three_card_combos)}\")\nprint(f\"Total 3-card combos calculation: C(5,3) = 5!/(3!·2!) =\", end=\" \")\nfrom math import comb\nprint(comb(5,3))"
    }
  },
  "result": "Total 3-card combos: 10\nTotal 3-card combos calculation: C(5,3) = 5!/(3!·2!) = 10"
}
```

</details>





Also 10 combos — because `C(5,3) = C(5,2) = 10`.

Here's an interesting poker question with this tiny deck: **You're dealt ♠A ♠K. Flop comes ♠Q, ♥A, ♦2. What's the probability your opponent has a better hand?**

They need a pair of Aces to beat your pair of Aces with King kicker — or did they? Let's see:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;# Your hand: ♠A ♠K\n# Board: ♠Q ♥A ♦2\n# R…&quot;)→&quot;[]&quot;</code></summary>

```json
{
  "id": "call_00_4KQIUunQ08ZtgozQABeJ4178",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "# Your hand: ♠A ♠K\n# Board: ♠Q ♥A ♦2\n# Remaining deck: [♠Q, ♥A, ♦2] are on board... wait, let me think about this differently.\n# Actually the remaining deck after dealing you ♠A ♠K is: [♠Q, ♥A, ♦2]\nyour_hand = [Card('♠','A'), Card('♠','K')]\nboard = [Card('♠','Q'), Card('♥','A'), Card('♦','2')]\nremaining = [c for c in mini_deck if c not in your_hand and c not in board]\nremaining"
    }
  },
  "result": "[]"
}
```

</details>





Actually, with only 5 cards total, there's nothing left for the opponent! Let me restructure — I'll show the connection between combinations and real poker with a **bigger mini scenario**:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;from math import comb\n\n# In a real 52-ca…&quot;)→&quot;Total 2-card hands from 52 cards: 1326\nP…&quot;</code></summary>

```json
{
  "id": "call_00_ypOs6376v2RJDJmdf1L43790",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "from math import comb\n\n# In a real 52-card deck:\ntotal_two_card_hands = comb(52, 2)\nprint(f\"Total 2-card hands from 52 cards: {total_two_card_hands}\")\n\n# How many of those are pocket Aces?\n# Need both Aces: C(4,2) = 6\npocket_aces = comb(4, 2)\nprint(f\"Pocket Aces combos: {pocket_aces}\")\nprint(f\"Probability: {pocket_aces}/{total_two_card_hands} = {pocket_aces/total_two_card_hands:.4%}\")\n\n# How many are any pair?\n# For each rank (13), choose 2 of the 4 suits\nany_pair = 13 * comb(4, 2)\nprint(f\"\\nAny pair combos: {any_pair}\")\nprint(f\"Probability: {any_pair/total_two_card_hands} = {any_pair/total_two_card_hands:.4%}\")\n\n# How many are suited?\n# For each suit (4), choose 2 of the 13 ranks\nsuited = 4 * comb(13, 2)\nprint(f\"\\nSuited combos: {suited}\")\nprint(f\"Probability: {suited/total_two_card_hands} = {suited/total_two_card_hands:.4%}\")"
    }
  },
  "result": "Total 2-card hands from 52 cards: 1326\nPocket Aces combos: 6\nProbability: 6/1326 = 0.4525%\n\nAny pair combos: 78\nProbability: 0.058823529411764705 = 5.8824%\n\nSuited combos: 312\nProbability: 0.23529411764705882 = 23.5294%"
}
```

</details>

Here's the pattern in summary:

| Event | # of combos | Probability |
|---|---|---|
| Any specific hand (e.g. A♠ K♠) | 1 | 0.075% |
| Pocket Aces (any suits) | 6 | 0.45% |
| Any pair | 78 | 5.88% |
| Suited hand | 312 | 23.53% |
| Any 2-card hand | 1,326 | 100% |

The key idea: **probability = (ways event can happen) / (total ways)**. And the 🧠

 trick is counting those ways with combinatorics — `comb(n, k)` = how many ways to pick k items from n items.

Want to explore a specific probability question for your poker program? Like the chance of flopping a flush with suited cards, or the odds of flopping a set with a pocket pair?

<details class='token-usage-details' markdown='1'><summary>$0.0093</summary>

`total=295,884 | in=293,858 | out=2,026 | cached=96.0% | reasoning=435 | $0.0093 | deepseek-v4-flash`

</details>


Can you create a small deck with five cards to teach and demonstrate me how probability and combinatorics work? Use python tool.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;# Create a small deck with 5 cards to te…&quot;)→&quot;[♠A, ♠K, ♠Q, ♥A, ♥K]&quot;</code></summary>

```json
{
  "id": "call_3eaab3c5038c4f1b86881c85",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "# Create a small deck with 5 cards to teach probability\nsmall_deck = [\n    Card('♠', 'A'),    # Ace of Spades\n    Card('♠', 'K'),    # King of Spades\n    Card('♠', 'Q'),    # Queen of Spades\n    Card('♥', 'A'),    # Ace of Hearts\n    Card('♥', 'K')     # King of Hearts\n]\nsmall_deck"
    }
  },
  "result": "[♠A, ♠K, ♠Q, ♥A, ♥K]"
}
```

</details>





I've created a small deck with 5 cards: ♠A, ♠K, ♠Q, ♥A, ♥K. This gives us a mix of ranks and suits to explore probability concepts.

With this deck, we can explore several probability and combinatorics ideas:
- What's the probability of drawing an Ace?
- What's the probability of drawing a Spade?
- How many ways can we choose 2 cards from these 5? (combinations)
- What's the probability of getting a pair (same rank)?
- What's the probability of getting a flush (same suit)?

Which concept would you like to explore first? Or do you have another idea?

<details class='token-usage-details' markdown='1'><summary>$0.0008</summary>

`total=81,611 | in=80,835 | out=776 | cached=99.3% | reasoning=519 | $0.0008 | mimo-v2.5`

</details>


## Player

`Player`class:

- balance: int
- bet: int
- playing: bool
- hand: List[Card] methods:
- take_action(round)
- call(round)
- check(round)
- fold(round)
- raise(round, amount).

In [ ]:
from enum import Enum, auto

class Action(Enum):
    RAISE = auto()
    CALL = auto()
    CHECK = auto()
    FOLD = auto()

### Strategy

`Player` has a `strategy` attribute, which chooses actions.

In [ ]:
def human(player, stage_bet): 
    prompt = f"""This is {player.username}'s turn! Type an action: Call, Check, Fold, Raise X with X amount
        Stage bet: {stage_bet}, your current bet: {player.bet}, your hand: {player.hand}"""
    while True:
        act, *amnt = input(prompt).split()
        if (action := getattr(Action, act.upper(), None)):
            return (action, stage_bet, int(*amnt) if amnt else 0)
        print(f"Invalid action: {act}. Try again.")

In [ ]:
def always_call(player, stage_bet): 
    print(f"Player {player.username} called with {player.hand}!")
    return (Action.CALL, stage_bet)

In [ ]:
class Player:
    def __init__(self, username, balance = 0, bet = 0, playing = True, hand = None, strategy = always_call):
        self.username = username                    # username
        self.balance = balance                      # total balance
        self.bet = bet                              # active bet this round
        self.playing = playing                      # playing this round or not
        self.hand = [] if hand is None else hand    # playing hand
        self.strategy = strategy                    # a strategy the player uses to make actions (human or ai)
    
    def perform_action(self, stage_bet) -> tuple[Action, int, int]:
        """Prompt player for an action, then call the appropriate function.
        To keep playing the round, the player has to match the `stage_bet`
        It returns a tuple of (`Action`, updated stage_bet, and player's bet)."""
        action, _, *amount = self.strategy(self, stage_bet)
        print(f'action: {action}, stage_bet: {stage_bet}, raise_amount: {amount}')
        return getattr(self, '_' + action.name.lower())(stage_bet, *amount)

    def _call(self, stage_bet, *args):
        """Player matches the current bet"""
        need = stage_bet - self.bet
        if self.balance < need:
            print(f"Cannot call. balance ({self.balance}) too low to match the stage_bet ({stage_bet}). Current bet: {self.bet}")
            return self.perform_action(stage_bet)
        else:
            self.balance -= need
            self.bet += need
            print(f'{self.username} called by betting {need}. Total self.bet: {self.bet}')
            return (Action.CALL, stage_bet, need)

    def _check(self, stage_bet, *args):
        """Player checks"""
        print(f'{self.username} checked')
        return (Action.CHECK, stage_bet, 0)

    def _fold(self, stage_bet, *args):
        """Player folds"""
        print(f'{self.username} folded')
        self.playing = False
        return (Action.FOLD, stage_bet, 0)

    def _raise(self, stage_bet, amount, *args):
        """Player raises by amount, and update `stage_bet`"""
        new_stage_bet = stage_bet + amount
        need = new_stage_bet - self.bet
        if need > self.balance:
            print(f'Cannot raise by {amount}. Your balance: {self.balance}, current bet: {self.bet}, stage_bet: {stage_bet}')
            return self.perform_action(stage_bet)
        else:
            print(f'{self.username} raised by {amount} to stage_bet: {new_stage_bet}!')
            self.balance -= need
            self.bet += need
            return (Action.RAISE, new_stage_bet, need)

    def __repr__(self): return f"Player username: {self.username}, balance: {self.balance}, bet: {self.bet}, hand: {self.hand}, playing: {self.playing}, human: {"Y" if self.strategy == human else "N"}"

In [ ]:
stage_bet = 2
player1 = Player('galopy', balance=20, playing=True, strategy=always_call)
act, stage_bet, bet = player1.perform_action(stage_bet)

Player galopy called with []!


action: Action.CALL, stage_bet: 2, raise_amount: []


galopy called by betting 2. Total self.bet: 2


In [ ]:
stage_bet, bet, player1

(2,
 2,
 Player username: galopy, balance: 18, bet: 2, hand: [], playing: True, human: N)

`Player` class works as intended, but can I make improvements on it? What would Norvig do?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good question. Let me look at what's here and point out a few things that Norvig's principles (clarity, simplicity, DRY, separation of concerns) might flag.

Three things jump out:

**1. The recursive fallback calls are broken.** In `_call` and `_raise`, when the player can't afford the bet, the code calls `self.perform_action()` with no arguments — but `perform_action` requires `stage_bet`. That'll throw a `TypeError` at runtime. This is the kind of thing Norvig would catch with a test.

**2. `_call` and `_raise` share the same "pay the difference" logic.** Both compute `need = target - self.bet`, check `self.balance < need`, then update `self.balance -= need` and `self.bet += need`. That's a DRY violation — a small helper like `_pay(need)` could clean it up.

**3. The `perform_action` unpacking is confusing.** `self.strategy(self, stage_bet)` returns `(Action.CALL, stage_bet)`, but that second value (the stage_bet from the strategy) gets thrown into `_` and discarded, while the original `stage_bet` parameter (not the one from the strategy) gets forwarded to `_call`/`_raise`. It's unclear which `stage_bet` matters and why the strategy returns one at all.

Which of these would you like to dig into first?

<details class='token-usage-details' markdown='1'><summary>$0.0055</summary>

`total=46,806 | in=45,401 | out=1,405 | cached=97.3% | reasoning=1,087 | $0.0055 | deepseek-v4-pro`

</details>


I fixed the recursive fallback calls. Let's write tests first. Then we know it works and after refactoring, we know it still works.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great approach. Let's think through the test scenarios first, since the spec tells us what the contract is. What do we need to verify?

Here are the main cases I'd suggest covering:

**Basic actions — happy path:**
- `call`: player has enough to match `stage_bet` → balance drops, bet increases to `stage_bet`, returns `CALL`
- `check`: no cost, returns `CHECK`
- `fold`: sets `playing = False`, returns `FOLD`
- `raise`: balance drops by full raise amount, new `stage_bet` returned

**Edge cases:**
- `call` with insufficient balance → what should happen? Will `_call` fall back to `perform_action` and let them choose a different action? If so, how do you test an input loop?
- `raise` with insufficient balance → same question
- `call` when already matched (`self.bet == stage_bet`) — should it do nothing?
- `call` when partially matched (`self.bet < stage_bet`) — only pay the difference
- `raise 0` — degenerate, but worth testing

**State:**
- After folding, the player's balance should be unchanged, bet unchanged, but `playing` is `False`

Does that look right, and which would you like to start with? Also — since `always_call` is predictable but `human` requires `input()`, which strategy will you use for the tests?

<details class='token-usage-details' markdown='1'><summary>$0.0036</summary>

`total=46,192 | in=45,811 | out=381 | cached=99.7% | reasoning=73 | $0.0036 | deepseek-v4-pro`

</details>


Actually before writing tests, does it make sense to have all those methods under `Player` class? Should I change something?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

That's a great instinct — better to get the design right before you're anchored by tests.

I see a tension in the current design: `Player` methods like `_call` and `_raise` modify `stage_bet` (game-level state) and return it. That means `Player` is reaching into the round's job. Your earlier note captured this perfectly — "This leads to circular dependency, which tightly couples these together. It becomes harder to test."

Norvig would likely ask: **what does each thing own?**

- **Player** owns: balance, bet, hand, playing status, and its strategy for choosing actions
- **Round** (or some game manager) owns: stage_bet, pot, enforcing the rules about what actions are valid when

One way to separate them: the `Player` just returns *what it wants to do* (the `Action` + amount), and the `Round` decides whether it's legal and updates all the state. That way:

- You can test `Player` in isolation (strategy, balance tracking)
- You can test the round logic independently (can they check? is the call amount right?)
- No circular dependency

Your `strategy` functions already do half of this — they return an action. The question is whether the `_call`/`_raise`/etc. logic should move out of `Player` and into the round.

What do you think — does splitting it that way appeal to you?

<details class='token-usage-details' markdown='1'><summary>$0.0051</summary>

`total=46,917 | in=46,200 | out=717 | cached=95.6% | reasoning=410 | $0.0051 | deepseek-v4-pro`

</details>


## Round

Round Management

Attributes:
- stage (pre-flop, flop, turn, river)
- players
- deck
- community_cards

Methods:

- `betting_round(round) -> None`: Manage one round of player actions (pre-flop, flop, turn, or river). Ends when all active players have called or checked.
- `distribute(pot, winners) -> None`: Distribute the pot to the winner(s).
- `start_round(round) -> None`: Orchestrate the whole hand: deal hole cards → blinds → betting round → flop → betting round → turn → betting round → river → betting round → showdown → distribute.

In [ ]:
class Stage(Enum):
    PREFLOP = auto()
    FLOP = auto()
    TURN = auto()
    RIVER = auto()

    def next(self):
        """Return the next stage, or None if already at RIVER."""
        return Stage(self.value + 1) if self.value < len(Stage) else None

In [ ]:
for s in Stage:
    print(s.next())

Stage.FLOP


Stage.TURN


Stage.RIVER


None


In [ ]:
cycle??


````python
def cycle(o):
    "Like `itertools.cycle` except creates list of `None`s if `o` is empty"
    o = listify(o)
    return itertools.cycle(o) if o is not None and len(o) > 0 else itertools.cycle([None])
````

**File:** `/usr/local/lib/python3.12/site-packages/fastcore/basics.py`; line: 706

In [ ]:
class Stage(Enum):
    PREFLOP = auto()
    FLOP = auto()
    TURN = auto()
    RIVER = auto()

    def next(self):
        """Return the next stage, or None if already at RIVER."""
        return Stage(self.value + 1) if self.value < len(Stage) else None

Let's go through playing a round of poker game!

In [ ]:
class Round:
    def __init__(self, players = None, deck = None, community = None, stage = Stage.PREFLOP, stage_bet = 0, pot = 0):
        self.players = [] if players is None else players
        self.deck = shuffle_deck([Card(s,r) for s in suits for r in ranks]) if deck is None else deck
        self.community = [] if community is None else community
        self.stage = stage
        self.stage_bet = stage_bet
        self.pot = pot
    
    def distribute(self, winners) -> None:
        "Distribute the prize pot to the winners"
        print(f"Distributing $ {self.pot} to {winners}")
        for player in winners:
            player.balance += self.pot // len(winners)

    def betting_round(self) -> None:
        "Manage one round of player actions (pre-flop, flop, turn, or river)"
        self.stage_bet = 0
        for p in self.players: p.bet = 0
        raiser, idx = -1, 0
        while idx != raiser:
            idx, raiser = self.player_act(idx, raiser)
            if raiser == -1: raiser = 0
    
    def player_act(self, idx, raiser):
        player = self.players[idx]
        if player.playing:
            act, stage_bet, player_bet = player.perform_action(self.stage_bet)
            self.pot += player_bet
            if act == Action.RAISE: 
                raiser = idx
                self.stage_bet = stage_bet
        idx = (idx + 1) % len(self.players)
        return idx, raiser

In [ ]:
@patch
def _showdown(self: Round):
    "Evaluate and distribute."
    winning_score = compare_hands([p.hand + self.community for p in self.players if p.playing])
    winners = [p for p in self.players if evaluate_hand(p.hand + self.community) == winning_score]
    print(f"Winners: {[p.username for p in winners]} with {winning_score}")
    self.distribute(winners)

In [ ]:
@patch
def _prep_round(self: Round):
    "Prepare for the round by getting the deck, shuffle, deal hole hands."
    self.deck = shuffle_deck([Card(s,r) for s in suits for r in ranks])
    for p in self.players: p.hand = withdraw_card(self.deck, 2)
    self.community = []
    self.pot = 0
    self.stage = Stage.PREFLOP

In [ ]:
@patch
def _reset_betting_round(self: Round):
    "Reset state for a new betting round."
    self._raiser, self.stage_bet, self._idx = -1, 0, 0
    for p in self.players: p.bet = 0

In [ ]:
@patch
def _start_betting_round(self: Round):
    if self.stage == Stage.FLOP: 
        self.community += withdraw_card(self.deck, 3)
    elif self.stage in (Stage.TURN, Stage.RIVER): 
        self.community += withdraw_card(self.deck)
    print(f"\n--- {self.stage.name} --- Community: {self.community}")

In [ ]:
@patch
def _get_next_player(self: Round) -> Player:
    while True:
        player = self.players[self._idx]
        if player.playing:
            if player.strategy is human:
                act, new_stage_bet, bet_amount = yield player
            else:
                act, new_stage_bet, bet_amount = player.perform_action(self.stage_bet)
            self.pot += bet_amount
            if act == Action.RAISE:
                self._raiser = self._idx
                self.stage_bet = new_stage_bet
        self._idx = (self._idx + 1) % len(self.players)
        if self._raiser == -1: self._raiser = 0
        if self._idx == self._raiser: break

In [ ]:
def play_action(action: str, round: Round, amount: int = 0):
    return getattr(player, '_' + action.name.lower())(round.stage_bet, amount)

In [ ]:
@patch
def start_round(self: Round):
    "Deal, start PREFLOP, and auto-advance to first human player."
    self._prep_round()
    self._reset_betting_round()
    for s in Stage:
        self.stage = s
        self._start_betting_round()
        self._get_next_player()
    self._showdown()

In [ ]:
@patch
def state(self: Round):
    "Return a string describing the current game situation."
    player = self.players[self._idx]
    return f"{player.username}'s turn | Stage: {self.stage.name} | Hand: {player.hand} | Community: {self.community} | Stage bet: {self.stage_bet} | Your bet: {player.bet} | Balance: {player.balance} | Pot: {self.pot}"

In [ ]:
@patch
def start_hand(self: Round) -> None:
    "Orchestrate the whole hand"
    # Get the board ready
    self.deck = shuffle_deck([Card(s,r) for s in suits for r in ranks])
    for p in self.players:
        p.hand = withdraw_card(self.deck, 2)
    # Betting stages
    for s in Stage:
        if s == Stage.FLOP:
            self.community = withdraw_card(self.deck, 3)
        elif s in (Stage.TURN, Stage.RIVER):
            self.community.extend(withdraw_card(self.deck))
        print(f"Starting stage {s}. Community cards: {self.community}")
        self.betting_round()
        print()
    # determine winners
    winning_score = compare_hands([p.hand + self.community for p in self.players if p.playing])
    winners = [p for p in self.players if evaluate_hand(p.hand + self.community) == winning_score]
    print(f"Winners are {winners} with score: {winning_score}")
    self.distribute(winners)

In [ ]:
player1 = Player('galopy', balance=20, playing=True, strategy=human)
player2 = Player('jalopy', balance=20, playing=True)
players = [player1, player2]
players

[Player username: galopy, balance: 20, bet: 0, hand: [], playing: True, human: Y,
 Player username: jalopy, balance: 20, bet: 0, hand: [], playing: True, human: N]

In [ ]:
round1 = Round(players = players)
round1.start_round()


--- PREFLOP --- Community: []



--- FLOP --- Community: [♥4, ♥3, ♠6]



--- TURN --- Community: [♥4, ♥3, ♠6, ♥10]



--- RIVER --- Community: [♥4, ♥3, ♠6, ♥10, ♠3]


Winners: ['galopy'] with (3, 4, 3, 10)


Distributing $ 0 to [Player username: galopy, balance: 20, bet: 0, hand: [♦4, ♦9], playing: True, human: Y]


In [ ]:
round1.state()

"galopy's turn | Stage: RIVER | Hand: [♦4, ♦9] | Community: [♥4, ♥3, ♠6, ♥10, ♠3] | Stage bet: 0 | Your bet: 0 | Balance: 20 | Pot: 0"

How does it look so far?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Looking great! You've built out all the major pieces following Norvig's approach — understand, specify, design, then implement. Let me highlight what's working well and one thing you might want to investigate:

**Working well:**
- `Card` as `namedtuple` with patched `__repr__`, `__lt__`, `__eq__` — clean and lightweight
- Hand evaluation with thorough tests (including edge cases like the wheel and two trips)
- `compare_hands` correctly returns the winning rank tuple
- Strategy pattern for players (`human` vs `always_call`) — nice separation
- Stage enum with `.next()` — clean progression

**One thing to look at:** your game ran through all four stages and reached a showdown, but the pot ended at **$0** and the human player (galopy) was never prompted. Can you trace through `start_round` and `_get_next_player` to see why the betting never happened?

Here's a hint — `_get_next_player` uses `yield`, which makes it a generator. How is `start_round` calling it?

<details class='token-usage-details' markdown='1'><summary>$0.0466</summary>

`total=45,572 | in=44,977 | out=595 | cached=60.9% | $0.0466 | glm-5p2`

</details>


ok so it just creates a generator but does not iterate it using `next` or a loop, right? I guess I can use `try except` block around `_get_next_player` to get next player.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Exactly — `start_round` just calls `self._get_next_player()` which creates the generator object but never iterates it, so nothing inside actually runs.

Your `try/except` idea is on the right track. Here's the thing to think through: the generator does two different jobs:

1. **For AI players** — it handles them internally (calls `perform_action` and moves on)
2. **For human players** — it `yield`s the player, waiting for you to `send` the action back

So when you get a `StopIteration`, that means the betting round is over (everyone has called or checked). When you get a yielded player, you need to interact with them.

How would you structure that loop? Think about what you need to do when you catch a yielded player vs when you catch `StopIteration`.

<details class='token-usage-details' markdown='1'><summary>$0.0129</summary>

`total=45,672 | in=45,304 | out=368 | cached=98.8% | $0.0129 | glm-5p2`

</details>


In [ ]:
@patch
def start_round(self: Round):
    "Deal, start PREFLOP, and auto-advance to first human player."
    self._prep_round()
    self._reset_betting_round()
    for s in Stage:
        self.stage = s
        self._start_betting_round()
        next_player_gen = self._get_next_player()
        while True:
            try:
                next_player = next(next_player_gen)
            except StopIteration:
                break
    self._showdown()

How does it look?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good progress — the `next()` and `StopIteration` pattern is exactly right for driving the generator. 

But there's a piece missing. Look back at `_get_next_player` — when it hits a human player, it does:

```python
act, new_stage_bet, bet_amount = yield player
```

That `yield` means the generator is pausing and **expecting you to send back** an action tuple. But right now your loop just calls `next(next_player_gen)` again, which sends `None` into the generator.

So for the human case, you'll need to do something different than just calling `next()`. What method on a generator lets you pass a value *back* into a `yield` expression?

<details class='token-usage-details' markdown='1'><summary>$0.0134</summary>

`total=46,173 | in=45,684 | out=489 | cached=99.3% | $0.0134 | glm-5p2`

</details>


I can use send. Is there a way to completely exit from the function so users can use solveit environment to play around and learn about important concepts in poker? 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great thinking — `send` is exactly the method. And your instinct about exiting is spot on. 

Here's the key insight: what if `start_round` itself were a **generator**? Instead of running the whole game in one shot, it could `yield` whenever it needs human input — pausing the entire game loop and returning control to the notebook.

Then in a code cell, you'd do something like:

```python
game = round1.start_round()
player = next(game)      # game pauses at first human turn
# ... user inspects state, plays around ...
action = (Action.CALL, round1.stage_bet, 0)
player = game.send(action)   # resume with their choice
```

Each `next` or `send` advances the game by one human action, and between calls the user has full access to the Python environment — they can check `round1.state()`, inspect hands, calculate odds, whatever they want.

Does that direction make sense to you?

<details class='token-usage-details' markdown='1'><summary>$0.0945</summary>

`total=46,379 | in=45,927 | out=452 | cached=5.6% | $0.0945 | glm-5p2`

</details>


In [ ]:
@patch
def start_round(self: Round):
    "Deal, start PREFLOP, and auto-advance to first human player."
    self._prep_round()
    self._reset_betting_round()
    for s in Stage:
        self.stage = s
        self._start_betting_round()
        yield from self._get_next_player()
    self._showdown()

In [ ]:
game = round1.start_round()
player = next(game)
player


--- PREFLOP --- Community: []


Player username: galopy, balance: 20, bet: 0, hand: [♣7, ♠J], playing: True, human: Y

In [ ]:
round1.state()

"galopy's turn | Stage: PREFLOP | Hand: [♣7, ♠J] | Community: [] | Stage bet: 0 | Your bet: 0 | Balance: 20 | Pot: 0"

In [ ]:
player = game.send(player._check(round1.stage_bet, 0))
player

galopy checked


Player jalopy called with [♦10, ♠4]!


action: Action.CALL, stage_bet: 0, raise_amount: []


jalopy called by betting 0. Total self.bet: 0



--- FLOP --- Community: [♥Q, ♦K, ♦J]


Player username: galopy, balance: 20, bet: 0, hand: [♣7, ♠J], playing: True, human: Y

In [ ]:
player = game.send(player._raise(round1.stage_bet, 2))
player

galopy raised by 2 to stage_bet: 2!


Player jalopy called with [♦10, ♠4]!


action: Action.CALL, stage_bet: 2, raise_amount: []


jalopy called by betting 2. Total self.bet: 2



--- TURN --- Community: [♥Q, ♦K, ♦J, ♣K]


Player username: galopy, balance: 18, bet: 2, hand: [♣7, ♠J], playing: True, human: Y

In [ ]:
round1.state()

"galopy's turn | Stage: TURN | Hand: [♣7, ♠J] | Community: [♥Q, ♦K, ♦J, ♣K] | Stage bet: 2 | Your bet: 2 | Balance: 18 | Pot: 4"

In [ ]:
player = game.send(player._raise(round1.stage_bet, 2))
player

galopy raised by 2 to stage_bet: 4!


Player jalopy called with [♦10, ♠4]!


action: Action.CALL, stage_bet: 4, raise_amount: []


jalopy called by betting 2. Total self.bet: 4



--- RIVER --- Community: [♥Q, ♦K, ♦J, ♣K, ♦3]


Player username: galopy, balance: 16, bet: 4, hand: [♣7, ♠J], playing: True, human: Y

In [ ]:
round1.state()

"galopy's turn | Stage: RIVER | Hand: [♣7, ♠J] | Community: [♥Q, ♦K, ♦J, ♣K, ♦3] | Stage bet: 4 | Your bet: 4 | Balance: 16 | Pot: 8"

In [ ]:
player = game.send(player._raise(round1.stage_bet, 2))
player

galopy raised by 2 to stage_bet: 6!


Player jalopy called with [♦10, ♠4]!


action: Action.CALL, stage_bet: 6, raise_amount: []


jalopy called by betting 2. Total self.bet: 6


Winners: ['galopy'] with (3, 13, 11, 12)


Distributing $ 12 to [Player username: galopy, balance: 14, bet: 6, hand: [♣7, ♠J], playing: True, human: Y]


StopIteration: 

In [ ]:
round1.state()

"galopy's turn | Stage: RIVER | Hand: [♣7, ♠J] | Community: [♥Q, ♦K, ♦J, ♣K, ♦3] | Stage bet: 6 | Your bet: 6 | Balance: 26 | Pot: 12"

It seems like it worked

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

It's working beautifully! The game flows through all four stages, the AI opponent calls automatically, the pot accumulates, and the showdown evaluates hands and distributes winnings. Galopy won with two pair (Kings and Jacks) — nice!

One small thing you might want to clean up: that `StopIteration` traceback at the end. It's not breaking anything — the game completed correctly and the pot was distributed — but it's a bit ugly. You could wrap your `game.send(...)` calls in a `try/except StopIteration` to catch it gracefully when the game is over.

Otherwise, the architecture is solid:
- Generator-based game loop gives you interactive control between turns
- `yield from` cleanly delegates to `_get_next_player`
- State is fully inspectable between actions via `round1.state()`
- Hand evaluation and pot distribution work correctly

What's next on your roadmap — blinds, the TODO items, or something else?

<details class='token-usage-details' markdown='1'><summary>$0.0502</summary>

`total=48,000 | in=47,735 | out=265 | cached=57.4% | $0.0502 | glm-5p2`

</details>


Yeah do I have to wrap 

```python
player = game.send(player._raise(round1.stage_bet, 2))
player
```

with try except block like this:

```python
try:
    player = game.send(player._raise(round1.stage_bet, 2))
except StopIteration:
    player = None
player
```



##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes, that's exactly it. The `StopIteration` means the game loop finished — showdown happened, pot was distributed, nothing left to do. Setting `player = None` is a clean way to signal "no more human turns."

<details class='token-usage-details' markdown='1'><summary>$0.0307</summary>

`total=49,919 | in=49,734 | out=185 | cached=6.7% | reasoning=135 | $0.0307 | deepseek-v4-pro`

</details>


Is there any way I can improve the user interface? It doesn't seem very convenient type all that out. Can I define a function?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Absolutely. You could define a small helper function that takes care of the ceremony — calling the right action method, sending it to the generator, and catching `StopIteration`. Something like:

```python
def act(game, player, round, action, amount=0):
    ...
```

Then your turn becomes something like:

```python
act(game, player, round1, Action.RAISE, 2)
```

How would you design what goes inside that function? Think about:
- Mapping `action` to the right `player._*` method
- `game.send(...)` with the result
- Catching `StopIteration`
- What to return (the next player, or `None` if the game is over)

<details class='token-usage-details' markdown='1'><summary>$0.0087</summary>

`total=50,204 | in=49,868 | out=336 | cached=82.9% | reasoning=185 | $0.0087 | deepseek-v4-pro`

</details>


In [ ]:
def act(game, player, round, action, amount=0):
    

In [ ]:
action = (Action.RAISE, round1.stage_bet + 2, 2)
player = game.send(action)
player

Player jalopy called with [♥Q, ♥7]!


action: Action.CALL, stage_bet: 2, raise_amount: []


jalopy called by betting 2. Total self.bet: 2



--- TURN --- Community: [♠Q, ♦4, ♦10, ♥8]


Player username: galopy, balance: 20, bet: 0, hand: [♠6, ♥4], playing: True, human: Y

In [ ]:
round1.state()

"galopy's turn | Stage: TURN | Hand: [♠6, ♥4] | Community: [♠Q, ♦4, ♦10, ♥8] | Stage bet: 2 | Your bet: 0 | Balance: 20 | Pot: 4"

I am not sure. Would Norvig use generator approach? I am not sure this approach is as clean as I thought it would have been. What would other approaches look like?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great question — it's worth stepping back and thinking about the design before committing.

The current approach works, but the `raiser, idx = -1, 0` with the `if raiser == -1: raiser = 0` hack is a bit fragile. The core challenge is: **after someone raises, everyone else needs to act again.** How you model that determines the cleanliness.

Here are three approaches worth considering:

**1. Queue/deque approach** — Probably the cleanest conceptually. Keep a `deque` of players who still need to act. Pop the front player, they act. If they call or check, they're done (don't re-add). If they raise, re-add all other *active* players to the queue. When the queue is empty, the round is over. No index tracking, no `raiser` variable.

**2. Flag-based approach** — Track a `has_acted` flag per player. After a raise, reset everyone's flag. The round ends when all active players have `has_acted = True`. Simple state, explicit termination condition.

**3. Generator/cycle approach** — Use `itertools.cycle` over active players, yielding each in turn. Break out when all active bets match `stage_bet` and everyone has had a chance to act since the last raise. This is more "functional" but the termination logic can get tricky.

Norvig tends to favor the simplest thing that clearly works. The deque approach maps very naturally to the problem — "who still needs to act?" is literally the queue contents. It also avoids the circular indexing math entirely.

Which of these resonates with you? Or would you like to explore one in more detail?

<details class='token-usage-details' markdown='1'><summary>$0.0892</summary>

`total=42,850 | in=41,930 | out=920 | cached=6.2% | $0.0892 | glm-5p2`

</details>


I am not sure. Would Norvig use generator approach? I am not sure this approach is as clean as I thought it would have been. What would other approaches look like?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good question to step back and reflect on. Let's look at a few approaches.

Your current approach uses a `while idx != raiser` loop with mutable state. It works, but you're right to ask whether there's a cleaner way.

Here are three alternative approaches for the betting round:

---

**1. Generator approach**

A generator yields control back to the caller after each action, letting an outer loop drive the game:

```python
def betting_round(self):
    self.stage_bet = 0
    for p in self.players: p.bet = 0
    raiser = -1
    idx = 0
    while idx != raiser:
        if raiser == -1: raiser = 0
        player = self.players[idx]
        if not player.playing:
            idx = (idx + 1) % len(self.players)
            continue
        yield player  # <-- caller gets the player, performs action, updates state
        idx = (idx + 1) % len(self.players)
```

Caller:
```python
for player in round.betting_round():
    act, new_stage_bet, bet = player.perform_action(round.stage_bet)
    round.pot += bet
    if act == Action.RAISE:
        round.stage_bet = new_stage_bet
        # need to communicate raiser index back somehow — this is the tricky bit
```

The generator is elegant in theory, but communicating the raiser index *back* into the generator is awkward. Norvig *did* use generators in cs212, but usually when the flow is one-directional.

---

**2. Explicit action queue (event-driven)**

Each action produces the next state explicitly:

```python
def betting_round(self):
    self.stage_bet = 0
    for p in self.players: p.bet = 0
    actor_idx = self._next_active(0)  # first active player
    last_raiser = actor_idx
    
    while True:
        player = self.players[actor_idx]
        act, new_stage_bet, bet = player.perform_action(self.stage_bet)
        self.pot += bet
        
        if act == Action.RAISE:
            last_raiser = actor_idx
            self.stage_bet = new_stage_bet
        elif act == Action.FOLD:
            player.playing = False
        
        actor_idx = self._next_active((actor_idx + 1) % len(self.players))
        if actor_idx == last_raiser: break  # everyone acted
```

This separates the state update from the iteration logic, which is clearer.

---

**3. Norvig-style: separate "find actor" and "all acted?"**

Norvig tends to extract helper functions so the main loop reads cleanly:

```python
def betting_round(self):
    self.stage_bet = 0
    for p in self.players: p.bet = 0
    last_raiser = self.first_active()
    
    while True:
        player = self.next_player()
        yield self._handle(player)
        if self.done(last_raiser): break
```

Each helper does one thing, and the main loop is almost plain English.

---

Which of these directions feels most like what you want? Or do you want to sketch one out together before picking?

<details class='token-usage-details' markdown='1'><summary>$0.0290</summary>

`total=46,470 | in=45,452 | out=1,018 | cached=7.3% | reasoning=305 | $0.0290 | deepseek-v4-pro`

</details>


I am not sure. Would Norvig use generator approach? I am not sure this approach is as clean as I thought it would have been. What would other approaches look like?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

That's a good instinct. Norvig tends to favor explicitness and simplicity over cleverness — the generator is a bit "magical" and harder to debug.

Here's a simpler approach: **manual stepping with two methods.**

- `round.state()` — returns a string describing whose turn it is and the game situation (no action taken, just information)
- `round.play_action(action, amount)` — takes your decision and advances one player's turn (AI players auto-advance through themselves until the next human)

You'd use it like:

```python
# Cell 1: see what's happening
round.state()  # "galopy's turn (PREFLOP). Hand: ♠A ♠K..."

# Cell 2: act
round.play_action(Action.RAISE, 5)

# Cell 3: see next state (AI players already processed)
round.state()  # "galopy's turn (PREFLOP). Opponent raised to 10..."

# Cell 4: act
round.play_action(Action.CALL, 0)
```

Each cell does exactly one thing, `round` holds all the game state visibly, and there's no hidden loop or yield mechanism. You can inspect `round` at any point between cells — check `round.players`, `round.community`, or even run `evaluate_hand` on your cards.

The downside is you need to manage the stage transitions yourself (or `play_action` handles them automatically when a betting round ends). But that's a small bit of logic to add.

How does this feel compared to the generator approach?

<details class='token-usage-details' markdown='1'><summary>$0.0076</summary>

`total=50,919 | in=50,206 | out=713 | cached=88.5% | reasoning=379 | $0.0076 | deepseek-v4-pro`

</details>


What would be nice approach if I eventually want to expand it to more players? And those players all have their actions on their separate computers. And everyone waits until it is their turn.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The manual stepping approach actually maps beautifully to a multi-player setup. Think of it as a **server** holding the `Round` state, with each player connecting from their own computer. The server's job is simple:

1. Figure out whose turn it is
2. Send that player the game state
3. Wait for their action
4. Apply it, repeat

Two clean approaches:

**A web server** (like FastAPI) — each player hits an endpoint. The server runs `round.play_action(...)` when it receives a decision, then pushes the updated state to whoever's turn is next. solveit could even act as the server, since you already have a running Python kernel:

```
Player A (browser)  →  solveit kernel  →  Player B (browser)
         "your turn: ♠A ♠K..."         "opponent raised to 10..."
```

**A simpler option for now**: keep everything in one solveit dialog and use the manual stepping pattern. When you're ready for multiplayer, extract `Round` into a proper server. The key point is that `round.state()` and `round.play_action(...)` are already a clean API — they don't care whether the caller is a local cell or a remote HTTP request.

So the manual stepping approach isn't just simple for now — it's also the right foundation for multiplayer later. Want to move forward with that?

<details class='token-usage-details' markdown='1'><summary>$0.0046</summary>

`total=51,150 | in=50,633 | out=517 | cached=97.8% | reasoning=225 | $0.0046 | deepseek-v4-pro`

</details>


flesh it out

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Here are the changes to `Round`. You'll want to put these in new cells below your existing code.

First, some internal helpers:

```python
@patch
def _start_betting_round(self: Round):
    "Reset state for a new betting round."
    self.stage_bet = 0
    for p in self.players: p.bet = 0
    self._raiser = -1
    self._idx = 0
```

Then the auto-advance logic — processes AI/folded players until the next human turn or the stage ends:

```python
@patch
def _auto_advance(self: Round):
    "Advance through AI/folded players until next human or stage ends."
    if self._raiser == -1: self._raiser = 0
    while self._idx != self._raiser:
        player = self.players[self._idx]
        if player.playing:
            if player.strategy is human: return
            act, new_stage_bet, bet_amount = player.perform_action(self.stage_bet)
            self.pot += bet_amount
            if act == Action.RAISE:
                self._raiser = self._idx
                self.stage_bet = new_stage_bet
        self._idx = (self._idx + 1) % len(self.players)
    self._advance_stage()
```

Stage progression and showdown:

```python
@patch
def _advance_stage(self: Round):
    "Advance to next stage or showdown, dealing community cards as needed."
    stages = list(Stage)
    i = stages.index(self.stage)
    if i == len(stages) - 1: return self._showdown()
    self.stage = stages[i + 1]
    if self.stage == Stage.FLOP:      self.community = withdraw_card(self.deck, 3)
    elif self.stage in (Stage.TURN, Stage.RIVER): self.community.extend(withdraw_card(self.deck))
    print(f"\n--- {self.stage.name} --- Community: {self.community}")
    self._start_betting_round()
    self._auto_advance()

@patch
def _showdown(self: Round):
    "Evaluate and distribute."
    winning_score = compare_hands([p.hand + self.community for p in self.players if p.playing])
    winners = [p for p in self.players if evaluate_hand(p.hand + self.community) == winning_score]
    print(f"Winners: {[p.username for p in winners]} with {winning_score}")
    self.distribute(winners)
    self._done = True
```

And the two public methods you'll actually use in cells:

```python
@patch
def start_hand(self: Round):
    "Deal, start PREFLOP, and auto-advance to first human player."
    self.deck = shuffle_deck([Card(s,r) for s in suits for r in ranks])
    for p in self.players: p.hand = withdraw_card(self.deck, 2)
    self.community = []
    self.pot = 0
    self.stage = Stage.PREFLOP
    self._done = False
    print(f"\n--- {self.stage.name} --- Community: {self.community}")
    self._start_betting_round()
    self._auto_advance()

@patch
def play_action(self: Round, action: Action, amount: int = 0):
    "Apply `action` for the current player, then auto-advance through AI players."
    if getattr(self, '_done', False): return print("Hand is already over.")
    player = self.players[self._idx]
    _, new_stage_bet, bet_amount = getattr(player, '_' + action.name.lower())(self.stage_bet, amount)
    self.pot += bet_amount
    if action == Action.RAISE:
        self._raiser = self._idx
        self.stage_bet = new_stage_bet
    self._idx = (self._idx + 1) % len(self.players)
    self._auto_advance()

@patch
def state(self: Round):
    "Return a string describing the current game situation."
    if getattr(self, '_done', False): return "Hand is over."
    player = self.players[self._idx]
    return f"{player.username}'s turn | Stage: {self.stage.name} | Hand: {player.hand} | Community: {self.community} | Stage bet: {self.stage_bet} | Your bet: {player.bet} | Balance: {player.balance} | Pot: {self.pot}"
```

How it works: `start_hand()` deals and races through AI turns to the first human. Then each cell you call `round1.play_action(Action.CALL)` (or `RAISE`, `CHECK`, `FOLD`), and it applies your move then advances through any AI turns, stopping at the next human — or at showdown if the hand's over. In between, you can inspect `round1` freely.

Want to try it out?

<details class='token-usage-details' markdown='1'><summary>$0.0112</summary>

`total=56,875 | in=51,301 | out=5,574 | cached=98.1% | reasoning=4,457 | $0.0112 | deepseek-v4-pro`

</details>


In [ ]:
@patch
def start_round(self: Round):
    "Deal, start PREFLOP, and auto-advance to first human player."
    self.deck = shuffle_deck([Card(s,r) for s in suits for r in ranks])
    for p in self.players: p.hand = withdraw_card(self.deck, 2)
    self.community = []
    self.pot = 0
    self.stage = Stage.PREFLOP
    print(f"\n--- {self.stage.name} --- Community: {self.community}")
    self._start_betting_round()
    return self._auto_advance()

In [ ]:
@patch
def _showdown(self: Round):
    "Evaluate and distribute."
    winning_score = compare_hands([p.hand + self.community for p in self.players if p.playing])
    winners = [p for p in self.players if evaluate_hand(p.hand + self.community) == winning_score]
    print(f"Winners: {[p.username for p in winners]} with {winning_score}")
    self.distribute(winners)

In [ ]:
@patch
def _start_betting_round(self: Round):
    "Reset state for a new betting round."
    self._raiser, self.stage_bet, self._idx = -1, 0, 0
    for p in self.players: p.bet = 0

In [ ]:
@patch
def _auto_advance(self: Round):
    "Advance through AI/folded players until next human or stage ends."
    while True:
        player = self.players[self._idx]
        if player.playing:
            if player.strategy is human: return
            act, new_stage_bet, bet_amount = player.perform_action(self.stage_bet)
            self.pot += bet_amount
            if act == Action.RAISE:
                self._raiser = self._idx
                self.stage_bet = new_stage_bet
        self._idx = (self._idx + 1) % len(self.players)
        if self._raiser == -1: self._raiser = 0
        if self._idx == self._raiser: break
    self._advance_stage()

In [ ]:
@patch
def play_action(self: Round, action: Action, amount: int = 0):
    "Apply `action` for the current player, then auto-advance through AI players."
    player = self.players[self._idx]
    _, new_stage_bet, bet_amount = getattr(player, '_' + action.name.lower())(self.stage_bet, amount)
    self.pot += bet_amount
    if action == Action.RAISE:
        self._raiser = self._idx
        self.stage_bet = new_stage_bet
    self._idx = (self._idx + 1) % len(self.players)
    self._auto_advance()

In [ ]:
@patch
def _advance_stage(self: Round):
    "Advance to next stage or showdown, dealing community cards as needed."
    self.stage = self.stage.next()
    if self.stage is None: return self._showdown()
    elif self.stage == Stage.FLOP: self.community += withdraw_card(self.deck, 3)
    elif self.stage in (Stage.TURN, Stage.RIVER): self.community += withdraw_card(self.deck)
    print(f"\n--- {self.stage.name} --- Community: {self.community}")
    self._start_betting_round()
    return self._auto_advance()

In [ ]:
@patch
def state(self: Round):
    "Return a string describing the current game situation."
    player = self.players[self._idx]
    return f"{player.username}'s turn | Stage: {self.stage.name} | Hand: {player.hand} | Community: {self.community} | Stage bet: {self.stage_bet} | Your bet: {player.bet} | Balance: {player.balance} | Pot: {self.pot}"

Ok thanks for fleshing it out. So I can start round using `start_round` method. After setting the board, it uses `_auto_advance` to skips Ai/folded players. But in the method, when the `player.strategy is human`, return? Why return? Woudln't it just exit from the whole game? Oh wait that's what I want actually. But here is an idea. Instead of using return and bunch of different methods that repeat themselves a lot, how about I through an exception when it is a human player? After that I just call a function that does something to get back into the flow. Or maybe use a generator approach to stop and resume?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Those are both interesting ideas. Here's how they compare:

**Exception approach**: Throw `HumanTurnException` inside the loop, catch it at the top level. The game state lives in `self`, so you just call a resume method when the human acts. Downside: exceptions for normal control flow is a code smell — it makes debugging confusing (stack traces where nothing went wrong) and readers expect exceptions to mean errors.

**Generator approach**: `_auto_advance` becomes a generator that `yield`s when it hits a human turn. You'd `next()` to run until the pause, then `gen.send(action)` to feed the human's decision back in. Elegant, but you have to manage the generator object across cells, handle `StopIteration` when the hand ends, and it's harder to inspect mid-game.

**Current return approach**: Each method does one thing and returns when it needs to pause. The "repetition" you're noticing is `_auto_advance` being called from two places (`start_round` and `play_action`), but that's just two call sites — not really DRY violation.

Honestly, the return-based approach is the most debuggable and the most natural fit for the "cell-by-cell" workflow in solveit. The generator is the most elegant but adds hidden complexity. The exception approach works but is the least idiomatic.

Which direction feels right to you?

<details class='token-usage-details' markdown='1'><summary>$0.0203</summary>

`total=45,718 | in=45,033 | out=685 | cached=92.6% | $0.0203 | glm-5p2`

</details>


I like the generator approach. What would Norvig do in this situation?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Norvig actually does use generators in CS212 — for instance, in his game-playing code where you explore possible moves. He's pragmatic about it: he uses generators when the pause/resume pattern fits the problem naturally, not just for elegance.

In this case, the poker loop genuinely has a natural "pause and wait for input" point — that's what generators are good at. The alternative (return + multiple call sites) works but is essentially hand-rolling what a generator gives you for free.

One thing Norvig would care about: keep the generator logic simple and readable. If the generator body reads like "deal, loop through players, yield when it's a human, continue" — that's clean. If it becomes a tangle of `send()` values and state management, that's a sign to step back.

Want to go ahead and try refactoring `_auto_advance` into a generator?

<details class='token-usage-details' markdown='1'><summary>$0.0138</summary>

`total=45,840 | in=45,386 | out=454 | cached=98.5% | $0.0138 | glm-5p2`

</details>


Ok I will try to make this as clean and clear as possible

In [ ]:
@patch
def _auto_advance(self: Round):
    "Advance through AI/folded players until next human or stage ends."
    while True:
        player = self.players[self._idx]
        if player.playing:
            if player.strategy is human: act, new_stage_bet, bet_amount = yield player
            else: act, new_stage_bet, bet_amount = player.perform_action(self.stage_bet)
            self.pot += bet_amount
            if act == Action.RAISE:
                self._raiser = self._idx
                self.stage_bet = new_stage_bet
        self._idx = (self._idx + 1) % len(self.players)
        if self._raiser == -1: self._raiser = 0
        if self._idx == self._raiser: break
    self._advance_stage()

How does it look?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

A few things to think about:

- The `yield` returns the human `Player` object, which is nice — `play_action` will need to know which player it's acting for, and that's already right there.

- There's a question about who updates what. Right now `play_action` also updates `pot`, `_raiser`, `stage_bet`, and `_idx`. With your generator handling all that after the `yield` picks back up, `play_action` should *not* duplicate those updates — just call the internal action and `send()` the tuple in. Double-updating `pot` would inflate it, and double-advancing `_idx` would skip players.

- The trickier one: your generator calls `self._advance_stage()` at the bottom when a betting round ends. But `_advance_stage` currently calls `_auto_advance()` again — which means your generator would be trying to recursively call itself. Generators don't work that way. You'd need to restructure so `_auto_advance` only handles *one* betting round, and something at the top level loops through stages. Or have `_advance_stage` start a new generator.

- How would `start_round` kick things off? It needs to create the generator, prime it with `send(None)`, handle `StopIteration` if there are no human players, and store the generator somewhere for `play_action` to use.

Want to work through one of these?

<details class='token-usage-details' markdown='1'><summary>$0.0073</summary>

`total=53,692 | in=50,696 | out=2,996 | cached=99.7% | reasoning=2,675 | $0.0073 | deepseek-v4-pro`

</details>


In [ ]:
player1 = Player('galopy', balance=20, playing=True, strategy=human)
player2 = Player('jalopy', balance=20, playing=True)
players = [player1, player2]
players

[Player username: galopy, balance: 20, bet: 0, hand: [], playing: True, human: Y,
 Player username: jalopy, balance: 20, bet: 0, hand: [], playing: True, human: N]

In [ ]:
round1 = Round(players = players)
round_gen = round1.start_round()


--- PREFLOP --- Community: []


In [ ]:
player = next(round_gen)
player

Player username: galopy, balance: 20, bet: 0, hand: [♥10, ♥9], playing: True, human: Y

In [ ]:
player.hand               # inspect and decide

[♥10, ♥9]

In [ ]:
def play_action(action: str, round: Round, amount: int = 0):
    return getattr(player, '_' + action.name.lower())(round.stage_bet, amount)

In [ ]:
try:
    player = gen.send(play_action(Action.RAISE, round1, 3))
except StopIteration:
    pass
player

galopy raised by 3 to stage_bet: 3!


Player username: galopy, balance: 17, bet: 3, hand: [♥10, ♥9], playing: True, human: Y

In [ ]:
round1.state()

"galopy's turn | Stage: PREFLOP | Hand: [♥10, ♥9] | Community: [] | Stage bet: 0 | Your bet: 3 | Balance: 17 | Pot: 0"

In [ ]:
try:
    player = gen.send(play_action(Action.RAISE, round1, 3))
except StopIteration:
    pass
player

galopy raised by 3 to stage_bet: 3!


Player username: galopy, balance: 14, bet: 3, hand: [♣10, ♥9], playing: True, human: Y

In [ ]:
round1.state()

"galopy's turn | Stage: FLOP | Hand: [♦4, ♦9] | Community: [♥4, ♥3, ♠6] | Stage bet: 0 | Your bet: 3 | Balance: 14 | Pot: 6"

In [ ]:
try:
    player = gen.send(play_action(Action.CHECK, round1, 3))
except StopIteration:
    pass
player

galopy checked


Player username: galopy, balance: 14, bet: 6, hand: [♥2, ♦Q], playing: True, human: Y

In [ ]:
round1.state()

"galopy's turn | Stage: PREFLOP | Hand: [♥2, ♦Q] | Community: [] | Stage bet: 3 | Your bet: 6 | Balance: 14 | Pot: 6"

Why does it not move to the next stage?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Look at what `_advance_stage` does at the end — it calls `self._auto_advance()`. But now that `_auto_advance` is a generator, what does that call actually do? Does anyone iterate the result?

That's the key question to think about.

<details class='token-usage-details' markdown='1'><summary>$0.1250</summary>

`total=52,154 | in=48,721 | out=3,433 | cached=0.0% | $0.1250 | glm-5p2`

</details>


Ok so it seems that it is stuck somehow and does not want to go to the next stage. I don't think it is reaching to `_advance_stage` at all. I think it is stuck in the while loop inside `_auto_advance` somehow even though `self.idx` and `self._raiser` are both 0.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Here's the problem, and it's a subtle one. Look at `_advance_stage`:

```python
@patch
def _advance_stage(self: Round):
    ...
    yield from self._auto_advance()
```

The `yield from` makes `_advance_stage` a **generator function**. And in Python, calling a generator function doesn't execute any code — it just returns a generator object. So when `_auto_advance` calls `self._advance_stage()` at the end, that line does **nothing**. The stage never advances, cards never get dealt.

That's why you're stuck in PREFLOP.

The fix: `_advance_stage` shouldn't be a generator. Instead, it should advance the stage normally and then iterate the new `_auto_advance` generator, yielding from it — which means the *calling code* (the outer `_auto_advance`) needs to `yield from` it instead.

<details class='token-usage-details' markdown='1'><summary>$0.0253</summary>

`total=55,370 | in=52,299 | out=3,071 | cached=41.9% | reasoning=2,859 | $0.0253 | deepseek-v4-pro`

</details>


I don't understand. Can you provide me with simple examples to explain?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Sure. Here's the key distinction:

```python
# This is a generator function (has yield inside)
def g():
    print("inside g!")
    yield 1

# Calling it does NOT print anything — it just returns a generator object
gen = g()          # nothing printed, no code ran
next(gen)          # NOW "inside g!" prints, and 1 is yielded
# inside g!
# 1
```

And here's the problem in your code:

```python
# Your _advance_stage looks like:
def _advance_stage(self):
    # ... advance the stage ...
    yield from self._auto_advance()   # this line makes it a generator
```

Because of that `yield from`, `_advance_stage` is a **generator function**. So when `_auto_advance` calls `self._advance_stage()`:

```python
self._advance_stage()   # returns a generator object, does nothing else
```

No code inside `_advance_stage` runs — the stage never advances, no cards are dealt, and the caller never iterates the returned generator anyway.

A quick demo:

```python
def advance_stage():
    print("dealing cards...")
    yield from []   # makes this a generator

print("calling...")
advance_stage()                    # nothing!
print("done — see? 'dealing cards' never printed")
```

```
calling...
done — see? 'dealing cards' never printed
```

The fix: `_advance_stage` shouldn't have `yield from` at all. It should just advance the stage and then the *caller* iterates the next generator. Does that make it clearer?

<details class='token-usage-details' markdown='1'><summary>$0.0091</summary>

`total=53,000 | in=52,580 | out=420 | cached=83.5% | reasoning=50 | $0.0091 | deepseek-v4-pro`

</details>


In [ ]:
# This is a generator function (has yield inside)
def g():
    print("inside g!")
    yield 1

# Calling it does NOT print anything — it just returns a generator object
gen = g()          # nothing printed, no code ran

In [ ]:
next(gen)          # NOW "inside g!" prints, and 1 is yielded

inside g!


1

In [ ]:
def advance_stage():
    print("dealing cards...")
    yield from []   # makes this a generator

print("calling...")
advance_stage()                    # nothing!

calling...


<generator object advance_stage at 0x76ff6ef84700>

In [ ]:

print("done — see? 'dealing cards' never printed")

done — see? 'dealing cards' never printed


Ok it is just creating a generator. I need to call `next`. Do I yield that??

Ok thanks for fleshing it out. So I can start round using `start_round` method. After setting the board, it uses `_auto_advance` to skips Ai/folded players. But in the method, when the `player.strategy is human`, return? Why return? Woudln't it just exit from the whole game? Oh wait that's what I want actually. But here is an idea. Instead of using return and bunch of different methods that repeat themselves a lot, how about I through an exception when it is a human player? After that I just call a function that does something to get back into the flow.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Interesting idea! Using an exception to "pause" the game and yield control is actually a known pattern — sometimes called "resumable exceptions" or using exceptions for non-local control flow. In Python it's a bit unconventional though.

Here are the tradeoffs:

**Why your `return` already works.** `_auto_advance` returns when it hits a human. Since `_auto_advance` was the last thing called in `play_action` (or `start_round`), that also returns — and now you're back at the top level. The round sits there, paused, waiting for your next `play_action()` call. No game exit, just a clean pause.

**Why exceptions are usually avoided for this in Python.** Python's exception handling is designed for *errors*, not expected branching. Using it for normal control flow makes debugging harder (try/except blocks in the debugger get confusing) and it's slower. Most Python devs would raise an eyebrow at `raise HumanTurn` — it reads like something went wrong, when it's actually the happy path.

**Alternatives that address the repetition.** If the real pain is that `_auto_advance` and `play_action` repeat the same player-processing loop, you could pull that out into a single method. Something like:

- `_process_player(player)` — applies one player's action, updates pot/bet/stage_bet, returns `True` if the round continues
- `_auto_advance` just calls `_process_player` in a loop
- `play_action` calls `_process_player` once

That keeps the action logic in one place and the control flow (loop vs single step) separate.

What part of the current code feels most repetitive to you?

<details class='token-usage-details' markdown='1'><summary>$0.0194</summary>

`total=48,361 | in=47,363 | out=998 | cached=46.2% | reasoning=632 | $0.0194 | deepseek-v4-pro`

</details>


Yeah in the current code, `_auto_advance` and `play_action` repeat. And I would like to have one method that abstracts out the logic of the game into easily understandable pieces and just calls other methods in order. 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The common logic is: get a player, apply their action, update state, advance the index. You can pull that into a single method — let's call it `_process_player` — and have both `play_action` and `_auto_advance` call into it.

The shape would be:

- `_process_player(player, action, amount)` — applies one action (updates pot, bet, stage_bet, raiser), then advances `_idx`
- `play_action` — calls `_process_player` with the human's chosen action, then calls `_auto_advance` to skip through AI players
- `_auto_advance` — loops: if human → `return` (pause), otherwise gets action from strategy and calls `_process_player`, then checks if the betting round is done

That way the "apply action and update state" logic lives in exactly one place. Both the human path and the AI path funnel through it.

Want to try sketching `_process_player` yourself, or would you like me to write it?

<details class='token-usage-details' markdown='1'><summary>$0.0155</summary>

`total=50,001 | in=49,325 | out=676 | cached=60.7% | reasoning=453 | $0.0155 | deepseek-v4-pro`

</details>


Sketch out for me

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Here's the sketch. One method, `_process_player`, handles the common logic. `play_action` and `_auto_advance` both funnel through it:

```python
def _process_player(self, player, action, amount=0):
    # apply action, update pot + stage_bet + raiser
    # advance self._idx
    ...

def play_action(self, action, amount=0):
    player = self.players[self._idx]
    self._process_player(player, action, amount)   # apply human's action
    self._auto_advance()                           # then skip through AI players

def _auto_advance(self):
    while True:
        player = self.players[self._idx]
        if not player.playing:
            self._idx = (self._idx + 1) % len(self.players)
            continue
        if player.strategy is human:
            return                                  # pause — wait for next play_action
        action, new_stage_bet, amount = player.perform_action(self.stage_bet)
        self._process_player(player, action, amount)
        if self._idx == self._raiser:              # betting round over?
            break
    self._advance_stage()                          # move to next stage or showdown
```

Three methods, each with one clear job. The only question is what exactly goes into `_process_player` — which is just the 3-4 lines that currently appear in both places (update pot, bet, raiser, index).

Want to write the inside of `_process_player` yourself, or shall I?

<details class='token-usage-details' markdown='1'><summary>$0.0089</summary>

`total=50,065 | in=49,607 | out=458 | cached=82.6% | reasoning=120 | $0.0089 | deepseek-v4-pro`

</details>


I don't like how `play_action` calls `_auto_advance`. Is it possible to write code that resumes back where it came from?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

You're describing exactly what **generators** do. A generator can `yield` to pause mid-function, and the next call to `next()` resumes right where it left off — same local variables, same loop position, everything intact.

So instead of `play_action` calling `_auto_advance`, the whole betting round lives inside one generator method:

```python
def _betting_round(self):
    for player in self._players_acting():        # loops through active players
        if player.strategy is human:
            yield                                 # pause — wait for human input
        else:
            action, _, amount = player.perform_action(self.stage_bet)
            self._process_player(player, action, amount)
    self._advance_stage()
```

And you'd drive it from outside like:

```python
round.start_round()           # calls _betting_round, runs until first yield
round.play_action(Action.CALL)  # feeds in the human's move, resumes generator
```

No `_auto_advance` at all. The loop itself *is* the flow, and `yield` is the pause button. A helper like `_players_acting` could cycle through players starting from the current `_idx`, handling the raiser logic internally.

Does this feel closer to what you had in mind?

<details class='token-usage-details' markdown='1'><summary>$0.0040</summary>

`total=50,516 | in=50,032 | out=484 | cached=99.8% | reasoning=202 | $0.0040 | deepseek-v4-pro`

</details>


But it would be a while loop in `_betting_round` as players can have more than one turn at each stage?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Exactly right — in a betting round, players can act multiple times (if someone raises, everyone else needs to respond). It's a `while` loop, not a `for` loop.

The generator would look more like:

```python
def _betting_round(self):
    self._raiser, self._idx = -1, 0
    while self._idx != self._raiser:
        player = self.players[self._idx]
        if player.playing:
            if player.strategy is human:
                yield                    # pause for human input
            else:
                action, _, amount = player.perform_action(self.stage_bet)
                self._process_player(player, action, amount)
        self._idx = (self._idx + 1) % len(self.players)
    self._advance_stage()
```

The `while` loop keeps cycling through players until the betting round ends. `yield` pauses it mid-loop for human input, and `play_action` calls `next()` to resume. When the round ends, `_advance_stage` handles dealing new cards and starts the next stage (which is another iteration of the same generator pattern).

How does this feel compared to the current code?

<details class='token-usage-details' markdown='1'><summary>$0.0044</summary>

`total=50,789 | in=50,395 | out=394 | cached=98.0% | reasoning=139 | $0.0044 | deepseek-v4-pro`

</details>


In [ ]:
player1 = Player('galopy', balance=20, playing=True, strategy=human)
player2 = Player('jalopy', balance=20, playing=True)
players = [player1, player2]
players

[Player username: galopy, balance: 20, bet: 0, hand: [], playing: True, human: Y,
 Player username: jalopy, balance: 20, bet: 0, hand: [], playing: True, human: N]

In [ ]:
round1 = Round(players = players)
round1.start_round()


--- PREFLOP --- Community: []


In [ ]:
round1.state()

"galopy's turn | Stage: PREFLOP | Hand: [♥10, ♥9] | Community: [] | Stage bet: 0 | Your bet: 0 | Balance: 20 | Pot: 0"

ok what should I do this round?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

You've got ♥10 ♥9 — suited connectors. That's a solid hand that can make both straights and flushes. 

Since stage bet is 0 and you're first to act, you have two options:

- **Check** — `round1.play_action(Action.CHECK)` — see the flop for free
- **Raise** — `round1.play_action(Action.RAISE, 3)` — build the pot and put pressure on jalopy

What do you feel like doing?

<details class='token-usage-details' markdown='1'><summary>$0.0096</summary>

`total=54,208 | in=53,943 | out=265 | cached=81.9% | reasoning=155 | $0.0096 | deepseek-v4-pro`

</details>


Let's check

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;round1.play_action(Action.CHECK)&quot;)→&quot;galopy checked\nPlayer jalopy called with…&quot;</code></summary>

```json
{
  "id": "call_00_4ugOQNVBcf1UKgQmK8Be9240",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "round1.play_action(Action.CHECK)"
    }
  },
  "result": "galopy checked\nPlayer jalopy called with [♦10, ♠7]!\naction: Action.CALL, stage_bet: 0, raise_amount: []\njalopy called by betting 0. Total self.bet: 0\n\n--- FLOP --- Community: [♠10, ♠J, ♥3]"
}
```

</details>





Both checked preflop. Flop is out: ♠10, ♠J, ♥3. Let's see where you stand:



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;round1.state()&quot;)→&quot;galopy&#x27;s turn | Stage: FLOP | Hand: [♥10…&quot;</code></summary>

```json
{
  "id": "call_00_wKQkbA9Q7pgaWbi69BiV0323",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "round1.state()"
    }
  },
  "result": "galopy's turn | Stage: FLOP | Hand: [♥10, ♥9] | Community: [♠10, ♠J, ♥3] | Stage bet: 0 | Your bet: 0 | Balance: 20 | Pot: 0"
}
```

</details>





You flopped middle pair (♥10) with a gutshot straight draw (8 or Q fills it). Board has two spades too. Jalopy checked behind you preflop — so they called with whatever they had.

Stage bet is 0 again. Check, or raise?

<details class='token-usage-details' markdown='1'><summary>$0.0124</summary>

`total=163,315 | in=162,903 | out=412 | cached=98.7% | reasoning=227 | $0.0124 | deepseek-v4-pro`

</details>


Thanks for taking actions for me using `python` tool. However, in solveit environment, it is difficult to comprehend the tool usages at a glance as they are collapsed with `details`. From now on, can you make your code and output explicit for me whenever you use those tools by using `python` codeblock in markdown with code and output?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Got it — I'll include the code and output explicitly in my responses from now on. Here's what just happened:

```python
round1.play_action(Action.CHECK)
```

```
galopy checked
Player jalopy called with [♦10, ♠7]!
jalopy called by betting 0. Total self.bet: 0

--- FLOP --- Community: [♠10, ♠J, ♥3]
```

```python
round1.state()
```

```
galopy's turn | Stage: FLOP | Hand: [♥10, ♥9] | Community: [♠10, ♠J, ♥3] | Stage bet: 0 | Your bet: 0 | Balance: 20 | Pot: 0
```

You flopped middle pair (♥10) with a gutshot straight draw. Stage bet is 0 — check or raise?

<details class='token-usage-details' markdown='1'><summary>$0.0047</summary>

`total=55,013 | in=54,577 | out=436 | cached=98.3% | reasoning=247 | $0.0047 | deepseek-v4-pro`

</details>


Much better! What do you mean by gutshot straight draw?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

I should correct myself — looking more carefully at your cards, that's not actually a gutshot.

With ♥10 ♥9 and the board ♠10, ♠J, ♥3, your ranks are 9, 10, 10, J, 3. To make a straight you'd need **two** specific cards (like 8+Q, or Q+K, or 7+8). That's called a runner-runner straight draw, and it's quite weak.

What I should have described as a gutshot is when you're missing exactly **one** card for a straight, with four connected ranks already in hand. For example:

- You hold 9, 10, Q, K — you need a **J** → that's a gutshot (one specific rank)
- You hold 8, 9, 10, J — you need a **7 or Q** → that's an *open-ended* straight draw (two possible ranks)

So a gutshot has exactly 4 outs (one rank × 4 suits), while an open-ender has 8 outs. A runner-runner draw is much weaker (about a 4% chance of hitting by the river).

My apologies for the sloppy terminology earlier — your straight prospects here are thin. Your real value on this flop is the middle pair (♥10).

<details class='token-usage-details' markdown='1'><summary>$0.0056</summary>

`total=55,928 | in=54,833 | out=1,095 | cached=98.3% | reasoning=810 | $0.0056 | deepseek-v4-pro`

</details>


ok should I call then?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Stage bet is 0, so there's nothing to call — calling means matching an existing bet. Your options right now are **check** or **raise**.

Since you have middle pair (10s) with a weak kicker and thin straight prospects, checking is a reasonable play — but it depends on how you want to approach jalopy. Check or raise?

<details class='token-usage-details' markdown='1'><summary>$0.0043</summary>

`total=55,337 | in=55,178 | out=159 | cached=98.4% | reasoning=85 | $0.0043 | deepseek-v4-pro`

</details>


I raise by 2

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;round1.play_action(Action.RAISE, 2)&quot;)→&quot;galopy raised by 2 to stage_bet: 2!\nPlay…&quot;</code></summary>

```json
{
  "id": "call_00_hbPiExIppNJbtbXn2oJp2952",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "round1.play_action(Action.RAISE, 2)"
    }
  },
  "result": "galopy raised by 2 to stage_bet: 2!\nPlayer jalopy called with [♦10, ♠7]!\naction: Action.CALL, stage_bet: 2, raise_amount: []\njalopy called by betting 2. Total self.bet: 2\n\n--- TURN --- Community: [♠10, ♠J, ♥3, ♥K]"
}
```

</details>





```python
round1.state()
```



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;round1.state()&quot;)→&quot;galopy&#x27;s turn | Stage: TURN | Hand: [♥10…&quot;</code></summary>

```json
{
  "id": "call_00_Hq80rGvoZpVbxpaUzfOh8602",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "round1.state()"
    }
  },
  "result": "galopy's turn | Stage: TURN | Hand: [♥10, ♥9] | Community: [♠10, ♠J, ♥3, ♥K] | Stage bet: 0 | Your bet: 0 | Balance: 18 | Pot: 4"
}
```

</details>





You raised, jalopy called. Turn is ♥K. Your ♥10 ♥9 now has middle pair plus a gutshot straight draw (any Q makes 9-10-J-Q-K). Pot is at 4, stage bet reset to 0. Check or raise?

<details class='token-usage-details' markdown='1'><summary>$0.0119</summary>

`total=166,849 | in=166,508 | out=341 | cached=99.4% | reasoning=174 | $0.0119 | deepseek-v4-pro`

</details>


you forgot to write code and output look nicely for me.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

You're right, my apologies. Here's what happened:

```python
round1.play_action(Action.RAISE, 2)
```

```
galopy raised by 2 to stage_bet: 2!
Player jalopy called with [♦10, ♠7]!
jalopy called by betting 2. Total self.bet: 2

--- TURN --- Community: [♠10, ♠J, ♥3, ♥K]
```

```python
round1.state()
```

```
galopy's turn | Stage: TURN | Hand: [♥10, ♥9] | Community: [♠10, ♠J, ♥3, ♥K] | Stage bet: 0 | Your bet: 0 | Balance: 18 | Pot: 4
```

You raised to 2, jalopy called. Turn brings ♥K. Your ♥10 ♥9 now has middle pair with a gutshot — any Q makes 9-10-J-Q-K. Pot is 4, stage bet is 0. Check or raise?

<details class='token-usage-details' markdown='1'><summary>$0.0044</summary>

`total=55,978 | in=55,718 | out=260 | cached=98.6% | reasoning=30 | $0.0044 | deepseek-v4-pro`

</details>


how likely is it to get a straight?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

From your perspective, you've seen 6 cards (your ♥10 ♥9 plus the board ♠10 ♠J ♥3 ♥K). That leaves 46 unknown cards in the deck. You need any Queen — there are 4 of them. So:

```python
outs = 4           # four Queens in the deck
unknown = 52 - 6   # 46 cards you haven't seen
outs / unknown
```

```
0.08695652173913043   # about 8.7%
```

About 8.7% chance to hit the straight on the river. Not great odds — roughly 1 in 11.5. 

Compare that to your other option: you already have middle pair (10s). If the river is a 10, you have three of a kind (2 outs). If it's a 9, you have two pair (3 outs). Combined with the straight, you have about 9 outs total to improve — about 19.6%.

That's the poker math side. What would you like to do — check, or raise again?

<details class='token-usage-details' markdown='1'><summary>$0.0048</summary>

`total=56,458 | in=56,011 | out=447 | cached=98.3% | reasoning=214 | $0.0048 | deepseek-v4-pro`

</details>


Raise 3. Make sure to make the output look nicely for me.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;round1.play_action(Action.RAISE, 3)&quot;)→&quot;galopy raised by 3 to stage_bet: 3!\nPlay…&quot;</code></summary>

```json
{
  "id": "call_00_O7NIyja6nwbmJTK9tVfE8428",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "round1.play_action(Action.RAISE, 3)"
    }
  },
  "result": "galopy raised by 3 to stage_bet: 3!\nPlayer jalopy called with [♦10, ♠7]!\naction: Action.CALL, stage_bet: 3, raise_amount: []\njalopy called by betting 3. Total self.bet: 3\n\n--- RIVER --- Community: [♠10, ♠J, ♥3, ♥K, ♣9]"
}
```

</details>







<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;round1.state()&quot;)→&quot;galopy&#x27;s turn | Stage: RIVER | Hand: [♥1…&quot;</code></summary>

```json
{
  "id": "call_00_h8NhqJwW3wyGqR4GwYxu9717",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "round1.state()"
    }
  },
  "result": "galopy's turn | Stage: RIVER | Hand: [♥10, ♥9] | Community: [♠10, ♠J, ♥3, ♥K, ♣9] | Stage bet: 0 | Your bet: 0 | Balance: 15 | Pot: 10"
}
```

</details>





```python
round1.state()
```

```
galopy's turn | Stage: RIVER | Hand: [♥10, ♥9] | Community: [♠10, ♠J, ♥3, ♥K, ♣9] | Stage bet: 0 | Your bet: 0 | Balance: 15 | Pot: 10
```

River is ♣9 — you now have **two pair**: 10s and 9s. That's a solid hand. Possible draws on board: any Queen makes a straight (Q-10-J-K-9), and there are two spades for a flush draw. But jalopy would have needed to be holding those.

Stage bet is 0, pot is 10. Check, or raise again?

<details class='token-usage-details' markdown='1'><summary>$0.0118</summary>

`total=169,893 | in=169,482 | out=411 | cached=99.8% | reasoning=148 | $0.0118 | deepseek-v4-pro`

</details>


raise 5

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;round1.play_action(Action.RAISE, 5)&quot;)→&quot;galopy raised by 5 to stage_bet: 5!\nPlay…&quot;</code></summary>

```json
{
  "id": "call_00_5r5Pu7bPvtJojOHkByqH1277",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "round1.play_action(Action.RAISE, 5)"
    }
  },
  "result": "galopy raised by 5 to stage_bet: 5!\nPlayer jalopy called with [♦10, ♠7]!\naction: Action.CALL, stage_bet: 5, raise_amount: []\njalopy called by betting 5. Total self.bet: 5\nWinners: ['galopy'] with (3, 10, 9, 13)\nDistributing $ 20 to [Player username: galopy, balance: 10, bet: 5, hand: [♥10, ♥9], playing: True, human: Y]"
}
```

</details>



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;round1.state()&quot;)→&quot;Hand is over.&quot;</code></summary>

```json
{
  "id": "call_01_4uFNAg1r5A4aOlK3g0Uv5252",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "round1.state()"
    }
  },
  "result": "Hand is over."
}
```

</details>





```python
round1.play_action(Action.RAISE, 5)
```

```
galopy raised by 5 to stage_bet: 5!
Player jalopy called with [♦10, ♠7]!
jalopy called by betting 5. Total self.bet: 5
Winners: ['galopy'] with (3, 10, 9, 13)
Distributing $ 20 to [Player username: galopy, balance: 10, bet: 5, hand: [♥10, ♥9], playing: True, human: Y]
```

```python
round1.state()
```

```
Hand is over.
```

You won! Your two pair (10s and 9s) beat jalopy's pair of 10s (he had ♦10 ♠7 — his 7 paired nothing). The winning hand rank was `(3, 10, 9, 13)` — two pair, 10s over 9s, King kicker. You collect the $20 pot.

<details class='token-usage-details' markdown='1'><summary>$0.0086</summary>

`total=114,214 | in=113,849 | out=365 | cached=98.9% | reasoning=60 | $0.0086 | deepseek-v4-pro`

</details>


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()